## Provider Anomaly: LLM Summarization (v3.5, GPT-OSS)

### What v3.5 guarantees before a case note is saved
- **Every feature has a description.** The run stops if any model feature has no dictionary row (feature dictionary v2.1, exported to `feature_mapping.csv`).
- **The right features.** Features are ranked by Impact_Score inside each provider before anything is cut; held, not-narrated and not-applicable features are removed first; near-duplicates collapse to one line.
- **No model internals reach the LLM.** No scores, tier words, z-scores or importance-file header text in the prompt; tier labels are translated to plain words, unknown labels are reported.
- **Numbers come from Python.** Provider values, peer medians and percentiles are computed from the scoring files (same claim category) and handed to the LLM as finished sentences. QA flags any number in the note that is not in the prompt.
- **Schemes come from the dictionary, not from the narrative.** Scheme labels are derived from Scheme_Link on the provider's driver features; the narrative may only name those schemes.
- **Nothing hidden is published.** The model's reasoning channel is never saved as a case note; truncated generations are retried once, then fall back to a factual evidence-only note.
- **QA you can trust.** Text is Unicode-normalized before checks; QA flags scores, ML words, "billed", intensifiers, delivery claims, medical-necessity language, invented numbers and ratios, unsupported schemes, and a wrong opening.
- **Audit trail.** The exact prompt, raw model output, token use and QA result for every provider are saved next to the summaries.

### Changelog (v3.5)
- Wired dictionary v2.1 fields (Display_Label, Description, Suspicious_Pattern, Benign_Explanations, Narrative_Caveat, Value_Template; Narrate_Role, Validation_Status, Applies_To, Concept_Group, Value_Type, Scheme_Link for code).
- Boundary-aware feature matching (`root` or `root_`), suffixes allowed only for one-hot roots.
- Main patterns = top 3 driver features; supporting drivers and background (context-only) items are separate blocks.
- Provider profile and peer comparison show the provider's own figures with peer median and percentile; claim lines and claims are no longer confused.
- Deterministic scheme labels (`derive_schemes`); the second LLM pass is removed.
- Harmony parsing uses `Role.ASSISTANT` and reads `TextContent.text`; reasoning effort is passed to the chat template.
- Hugging Face token read from an environment variable or Databricks secret.

### Earlier versions
v3.4 scheme definitions and rationale; v3.2 GPT-OSS with Harmony; v3.1 code decoding of procedure, diagnosis and modifier codes in feature names.


##### Dependencies

In [ ]:
%pip install torch accelerate sentencepiece openai-harmony 
%pip install huggingface_hub==0.23.0
%pip install transformers==5.2.0 --force-reinstall
dbutils.library.restartPython()

In [ ]:
import os
import re
import ast
import sys
import glob
import json
import math
import hashlib
import unicodedata
import joblib
import warnings
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from huggingface_hub import snapshot_download, login
from collections import Counter, defaultdict
from datetime import datetime, timedelta, date
from ast import literal_eval

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from pyspark.sql.functions import col

warnings.filterwarnings('ignore', category=UserWarning, module='threadpoolctl')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:,.6f}'.format


In [ ]:
CATEGORY_ALIASES = {
    'bh outpatient clinic': [
        'behavioral health outpatient clinic', 'behavioral health outpatient',
        'bh outpatient', 'behavioral-health outpatient',
    ],
    'bh residential treatment': [
        'behavioral health residential treatment', 'behavioral-health residential',
        'bh residential', 'behavioral health residential',
    ],
    'bh professional': [
        'behavioral health professional', 'bh mental health professional',
        'behavioral-health professional', 'bh mental health',
    ],
    'nemt not ihs': [
        'nemt-not-ihs', 'nemt not ihs', 'non-ihs nemt', 'nemt (not ihs)', 'nemt (non-ihs)', 'nemt non-ihs',
        'non-emergency medical transportation (not ihs)',
        'non-emergency medical transportation not ihs',
    ],
    'nemt ihs': [
        'nemt ihs', 'nemt-ihs', 'ihs-nemt', 'ihs nemt', 'nemt (ihs)',
        'non-emergency medical transportation ihs',
    ],
    'dme and appliances': [
        'dme and appliances', 'durable medical equipment and appliances',
        'durable medical equipment', 'dme appliances',
    ],
    'dme medical supplies': [
        'dme medical supplies', 'durable medical equipment supplies',
        'durable medical equipment medical supplies', 'dme supplies',
    ],
    'nursing home': [
        'nursing home', 'nursing home care', 'nursing facility',
        'skilled nursing', 'long-term care',
    ],
}

# Display form of each claim category inside Python-written value sentences
CATEGORY_DISPLAY = {
    'NEMT NOT IHS': 'NEMT (non-IHS)', 'NEMT IHS': 'NEMT (IHS)', 'DME MEDICAL SUPPLIES': 'DME medical supplies',
    'DME AND APPLIANCES': 'DME and appliances', 'NURSING HOME': 'nursing home',
    'BH OUTPATIENT CLINIC': 'BH outpatient clinic', 'BH RESIDENTIAL TREATMENT': 'BH residential treatment',
    'BH PROFESSIONAL': 'BH professional',
}

# ── Evidence strength display mapping ──
EVIDENCE_DISPLAY = {
    'high': 'strong evidence',
    'medium': 'moderate evidence',
    'low': 'limited evidence',
    'none': 'not assessed',
}

# ── Scheme display name for "other" (investigator-facing: no model vocabulary) ──
SCHEME_DISPLAY_NAMES = {
    'other': 'No known scheme (possible new pattern)',
}


In [ ]:
# GPT-OSS writes narrow no-break spaces (U+202F), non-breaking hyphens (U+2011), curly quotes, and sometimes
# U+202F as a thousands separator ("10 655"). Every text check runs on normalized text.
_UNI_MAP = str.maketrans({
    **{chr(c): '-' for c in (0x2010, 0x2011, 0x2012, 0x2013, 0x2014, 0x2015, 0x2212)},
    **{chr(c): ' ' for c in (0x00a0, 0x2007, 0x2009, 0x202f)},
    chr(0x200b): '',
    chr(0x2018): "'", chr(0x2019): "'", chr(0x201c): '"', chr(0x201d): '"', chr(0x00d7): 'x',
})
_DIGIT_GROUP_SPACE = re.compile('(?<=\\d)[' + chr(0x00a0) + chr(0x2007) + chr(0x2009) + chr(0x202f) + '](?=\\d{3}(?!\\d))')


def normalize_text(s):
    """Thousands-separator spaces to commas, NFKC, then dashes, special spaces and curly quotes to ASCII."""
    if s is None:
        return ''
    s = _DIGIT_GROUP_SPACE.sub(',', str(s))
    return unicodedata.normalize('NFKC', s).translate(_UNI_MAP)


def _norm_words(s):
    return ' ' + re.sub(r'[^a-z0-9]+', ' ', normalize_text(s).lower()).strip() + ' '


def category_mentioned(summary, category):
    """True if the claim category (or an alias) appears as whole words, after normalization.
    Longer names of other categories are removed first, so 'durable medical equipment supplies'
    does not count as 'durable medical equipment' (DME AND APPLIANCES)."""
    s = _norm_words(summary)
    cat = str(category).lower().strip()
    own = [_norm_words(a) for a in [cat] + CATEGORY_ALIASES.get(cat, [])]
    for other, aliases in CATEGORY_ALIASES.items():
        if other == cat:
            continue
        for a in sorted([other] + aliases, key=len, reverse=True):
            na = _norm_words(a)
            if any(o.strip() in na.strip() for o in own) and na not in own:
                s = s.replace(na, ' ')
    if cat == 'nemt ihs' and re.search(r' no[nt] ihs ', s):
        return False                      # 'non-IHS NEMT' / 'NEMT not IHS' is the other category
    return any(o in s for o in own)


def normalize_provider_id(series):
    """Provider IDs as clean strings ('1029', not 1029.0); blanks become NaN."""
    s = series.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return s.where(~s.isin(['', 'nan', 'None', 'NaN', '<NA>']))


##### Configuration

In [ ]:
provider_id = 'Servicing_Provider_ID'
ROOT = 'sedo.senou@azahcccs.gov'

with open(
    f'/Workspace/Users/{ROOT}/Projects/all_projects_predictions/data/scoring_date.txt', 'r') as f:
    scoring_date = f.read().strip()

master_table_llm_summarization = "main.prod_results.master_anomaly_detection_referrals_and_llm_summarization"
test_table_llm_summarization = "main.sedo.test_anomaly_detection_referrals_and_llm_summarization"

NOTEBOOK_VERSION = 'v3.5'
DICTIONARY_VERSION = 'v2.1'
# Optional: path to AHCCCS_feature_dictionary_v2.1.xlsx. When set, cell 31 regenerates feature_mapping.csv from it.
DICTIONARY_XLSX_PATH = None

# ── Model configuration ──
#   gpt-oss-120b: 117B params (5.1B active), single 80GB GPU (H100/A100/MI300X)
#   gpt-oss-20b:  21B params (3.6B active), fits 16GB GPU
MODEL_ID = "openai/gpt-oss-120b"
# MODEL_ID = "openai/gpt-oss-20b"

# Reasoning effort is passed to the chat template (reasoning_effort=...). "low" keeps the hidden
# reasoning short so the token budget reaches the final answer.
REASONING_LEVEL = "low"

# Hugging Face token: read from env var HF_TOKEN, else from this Databricks secret (fill in scope/key).
HF_SECRET_SCOPE = None
HF_SECRET_KEY = 'hf_token'

# ── Narrative settings ──
TOP_NUM = 20                 # driver features considered per provider (after filtering and de-duplication)
TIER1_DRIVERS = 3            # MAIN PATTERNS shown to the LLM; equals the 3-pattern limit of the note
MAX_PATTERNS = 3
MAX_CONTEXT_ITEMS = 4        # background (context-only) items shown
MAX_NEW_TOKENS = 800
RETRY_MAX_NEW_TOKENS = 1600  # one retry when the final answer is missing or cut off
MIN_CLAIMS_FOR_VALUE = 20    # below this, a provider value rests on too few claims to quote
DRIVER_PRIORITY_RATIO = 0.8  # in a concept group, a Driver outranks a stronger Context item only within 80% of it
RELATED_MIN_RATIO = 0.2      # a sibling counts as a "related measure" only if it scored >= 20% of the kept one
SCHEME_PCT_HIGH = 75         # a scheme is linked to a High-concern feature only at or above this peer percentile
SCHEME_PCT_LOW = 25          # ... and to a Low-concern feature only at or below this one

# ── Scheme definitions: plain, paid-claims wording; no intent, no medical-necessity judgment ──
# Only labels that claims data can support are defined. The narrative may name a scheme only when the
# provider's driver features link to it (dictionary Scheme_Link), and only as "consistent with".
SCHEME_DEFINITIONS_BY_LABEL = {
    'upcoding': 'payment for a more expensive code than peers are paid for with similar patients, or for a diagnosis rarely paired with that equipment or service among peers',
    'unbundling': 'payment for several line items or claims where peers are paid for one combined service',
    'modifier misuse': 'payment with modifiers that change reimbursement (for example RR = rental) in combinations peers rarely use',
    'diagnosis inflation/padding': 'unspecified or rarely seen diagnoses on paid claims far more often than at peers',
    'excessive visits': 'far more paid encounters per patient than peers have for the same type of service',
    'frequency abuse': 'paid units or services above a policy frequency or quantity limit, or far above peers for the same code',
    'phantom billing': 'paid services with no supporting activity in the claims data, such as trips with no medical claim for the member that day; this is a gap to check, not proof the service did not happen',
    'duplicate billing': 'the same service paid more than once for the same patient and date',
    'service upcreep': 'a shift toward higher-cost codes over time, faster than peers serving similar patients',
    'place-of-service manipulation': 'payment with a place of service that peers rarely pair with the procedure',
    'billing for non-covered services': "payment for services outside the member's covered benefit or eligibility, or for non-covered trip purposes; weekend or holiday dates are a timing pattern, not non-covered",
    'high-cost outlier manipulation': 'payments concentrated in a few high-cost codes or claims far above peers',
    'impossible day billing': 'time-based units for one patient on one day that add up to more hours than a day holds',
    'rental vs purchase manipulation': 'rental and purchase payments for the same equipment in patterns peers rarely show',
    'mileage inflation': 'paid transport miles or wait-time units far above what peers are paid for similar trips',
    'geographic improbability': 'trip distances or patient locations that are implausible for the service area',
    'level of care upcoding': 'payment at a higher level of care than peers use for similar patients',
    'group therapy upcoded to individual': 'individual therapy codes paid where peers mostly use group codes',
}
SCHEME_DEFINITIONS = "\n".join(f"- {k}: {v}" for k, v in SCHEME_DEFINITIONS_BY_LABEL.items())

# ── Known Medicaid billing scheme labels (unchanged list, used by downstream reports) ──
KNOWN_SCHEMES = [
    "upcoding", "unbundling", "excessive visits", "frequency abuse", "phantom billing",
    "duplicate billing", "service upcreep", "place-of-service manipulation",
    "modifier misuse", "medically unnecessary services",
    "billing for non-covered services", "high-cost outlier manipulation",
    "impossible day billing", "ineligible hospice enrollment",
    "medically unnecessary dme", "rental vs purchase manipulation",
    "mileage inflation", "geographic improbability",
    "level of care upcoding", "group therapy upcoded to individual",
    "diagnosis inflation/padding", "other",
]


##### Helper Functions

In [ ]:
def _harmony_text(content):
    """openai_harmony Message.content is a list of TextContent objects; return their text."""
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return ''.join(_harmony_text(c) for c in content)
    text = getattr(content, 'text', None)
    return text if isinstance(text, str) else ''


def generate_with_harmony(messages, max_new_tokens=MAX_NEW_TOKENS, return_meta=False):
    """
    Generate with model.generate and split GPT-OSS output into Harmony channels.

    Returns (final_text, analysis_text), or (final_text, analysis_text, meta) with return_meta=True.
    final_text is '' when the model never reached the final channel. analysis_text is the hidden
    reasoning: for debugging only, never published.
    meta: new_tokens, hit_token_limit, final_found, parse ('harmony' or 'fallback (...)').
    """
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True,
        reasoning_effort=REASONING_LEVEL,
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_token_ids = output_ids[0][input_ids.shape[1]:].tolist()
    meta = {'new_tokens': len(new_token_ids), 'hit_token_limit': len(new_token_ids) >= max_new_tokens}
    final_text, analysis_text = "", ""

    try:
        from openai_harmony import Role
        parsed_messages = harmony_enc.parse_messages_from_completion_tokens(new_token_ids, Role.ASSISTANT)
        for msg in parsed_messages:
            channel = str(getattr(msg, 'channel', '') or '')
            text = _harmony_text(getattr(msg, 'content', None))
            if channel == 'final':
                final_text = text
            elif channel == 'analysis':
                analysis_text += text + " "
        meta['parse'] = 'harmony'
    except Exception as e:
        # Truncated or malformed output: read the channels from the raw decoded text instead
        meta['parse'] = f'fallback ({type(e).__name__})'
        raw_text = tokenizer.decode(new_token_ids, skip_special_tokens=False)
        if '<|channel|>final<|message|>' in raw_text:
            final_part = raw_text.split('<|channel|>final<|message|>')[-1]
            for stop_token in ['<|return|>', '<|end|>', '<|start|>']:
                final_part = final_part.split(stop_token)[0]
            final_text = final_part
        if '<|channel|>analysis<|message|>' in raw_text:
            analysis_text = raw_text.split('<|channel|>analysis<|message|>', 1)[1].split('<|end|>')[0]

    final_text, analysis_text = final_text.strip(), analysis_text.strip()
    meta['final_found'] = bool(final_text)
    if return_meta:
        return final_text, analysis_text, meta
    return final_text, analysis_text


In [ ]:
# Sentence boundary that does not split on "vs.", "e.g." or "i.e."
_SENT = re.compile(r'(?<=[.!?])(?<!\bvs\.)(?<!e\.g\.)(?<!i\.e\.)\s+(?=[A-Z0-9"(])')


def clean_llm_summary(text):
    """Normalize characters, strip markdown and header words (only at line start), drop repeated sentences."""
    if not isinstance(text, str) or not text.strip():
        return ""
    text = normalize_text(text)
    text = re.sub(r'<\|[^|]+\|>', '', text)                                     # stray Harmony markers
    text = re.sub(r'(?m)^\s{0,3}(#{1,6}\s*|[-*•]\s+|\d+[.)]\s+)', '', text)  # headings, bullets, numbering
    text = text.replace('**', '').replace('__', '')
    text = re.sub(r'(?im)^\s*(summary|conclusion|in summary|in conclusion|answer|case note)\s*:\s*', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    paras, seen = [], set()
    for para in text.split('\n\n'):
        kept = []
        for s in _SENT.split(para.strip()):
            key = s.strip().lower().rstrip('.')
            if key and key not in seen:
                seen.add(key)
                kept.append(s.strip())
        if kept:
            paras.append(' '.join(kept))
    text = '\n\n'.join(paras)
    return re.sub(r'[ \t]+', ' ', text).strip()


def safe_literal_eval(x):
    try:
        if isinstance(x, (list, tuple)):
            return list(x)
        if pd.isna(x) or str(x).strip() in ['nan', 'None', '', '[]']:
            return []
        return literal_eval(str(x))
    except (ValueError, SyntaxError, TypeError):
        return []


def capitalize_two_letter_words_except(text, exceptions):
    return re.sub(
        r'\b[a-zA-Z]{2}\b',
        lambda x: x.group(0).upper() if x.group(0).lower() not in exceptions else x.group(0),
        text
    )


# ── Feature to dictionary matching ─────────────────────────────────────────────────────────────
# A feature matches a root when it equals the root, or (one-hot roots only) starts with root + '_'.
# Roots are tried longest first, so 'average_procedure_zscore_class_X' never matches 'average_procedure_zscore'.
# MAP, ROOTS_LONGEST_FIRST and SUFFIX_OK_ROOTS are built in cell 31 from feature_mapping.csv.

def match_root(feature):
    if feature is None or (isinstance(feature, float) and pd.isna(feature)):
        return ''
    s = str(feature).strip()
    for root in ROOTS_LONGEST_FIRST:
        if s == root or (root in SUFFIX_OK_ROOTS and s.startswith(root + '_')):
            return root
    return ''


get_mapping_root = match_root      # name used by the v2 wiring patch


def get_mapping_field(feature, col):
    return str(MAP.get(match_root(feature), {}).get(col, '') or '')


def _clean_code_level(raw):
    """'RR,N1,-1,-1__E1390' -> 'RR, N1 with E1390' (placeholders -1 and 0 removed)."""
    out = []
    for part in raw.split('__'):
        toks = [t.strip() for t in part.split(',') if t.strip() not in ('', '-1', '0')]
        if toks:
            out.append(', '.join(toks))
    return ' with '.join(out) if out else raw


def plain_level(feature):
    """Plain-English level carried in a one-hot feature name, or '' if none."""
    s, root = str(feature).strip(), match_root(feature)
    if not root or len(s) <= len(root):
        return ''
    raw = s[len(root) + 1:]
    key = raw.replace('_', ' ').strip().lower()
    vtype = MAP[root].get('Value_Type', '')
    if root in TIER_ROOTS:
        if key in TIER_PLAIN:
            return TIER_PLAIN[key]
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level'
    if root == 'unusual_wait_time_class':
        if key in WAIT_BANDS:
            return WAIT_BANDS[key]
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual wait-time band'
    if root in ('visit_frequency_class', 'weeks_with_unusual_visits_class'):
        m = re.search(r'above\s*(\d+)', key)
        if m:
            return f"top {100 - int(m.group(1))}% of providers"
        if 'below' in key:
            return 'below the top 5% of providers'
    if vtype == 'Yes/No flag':
        if key in ('1', '1.0', 'true', 'y', 'yes'):
            return ''
        if key in ('0', '0.0', 'false', 'n', 'no'):
            return 'not present'
    if vtype == 'Tier label' and RISKY_LEVEL_WORDS.search(key):
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level'
    if vtype == 'Category level' and ('__' in raw or ',' in raw):
        return _clean_code_level(raw)
    return raw.replace('_', ' ').strip()


def simplify_feature_name(feature):
    if pd.isna(feature):
        return None
    root = match_root(feature)
    if not root:                                   # never reaches the LLM: cell 33 stops on unmapped features
        exceptions = {'of', 'to', 'or', 'at', 'on', 'no', 'by', 'in'}
        return capitalize_two_letter_words_except(str(feature).replace("_", " ").title(), exceptions)
    label = mapping_dict.get(root, root)
    level = plain_level(feature)
    return f"{label}: {level}" if level else label


def get_feature_purpose(feature):
    return get_mapping_field(feature, 'Purpose') if not pd.isna(feature) else None


def get_clinical_context(feature):
    return get_mapping_field(feature, 'Clinical_Context') if not pd.isna(feature) else ""


def get_thresholds(feature):
    return get_mapping_field(feature, 'Thresholds') if not pd.isna(feature) else ""


# Codes shown to the LLM keep the code and add a description. Codes without any description are counted
# (CODES_WITHOUT_DESCRIPTION) and reported in the trust report.
CODES_WITHOUT_DESCRIPTION = Counter()
CODES_DESCRIBED_BY_GROUP = Counter()


def _cap(s):
    s = str(s).strip()
    return s[:1].upper() + s[1:] if s else s


def describe_diag(code):
    """'F4312' or 'F43.12' -> exact description if the code table has it, else its 3-character ICD-10 group
    from icd_10_diagnosis_code_root_and_category.csv, labelled as a group so the LLM never treats it as exact."""
    c = str(code).strip().upper().replace('.', '')
    if not c:
        return ''
    desc = diag_code_lookup.get(c, '')
    if desc:
        return f"Diagnosis {c}: {_cap(desc)}"
    grp = diag_group_lookup.get(c[:3])
    if grp:
        CODES_DESCRIBED_BY_GROUP[('diagnosis', c)] += 1
        return f"Diagnosis {c} (group {c[:3]}: {_cap(grp)})"
    CODES_WITHOUT_DESCRIPTION[('diagnosis', c)] += 1
    return f"Diagnosis {c}"


def describe_proc(code):
    c = str(code).strip().upper()
    desc = proc_code_lookup.get(c, '')
    if desc:
        return f"Procedure {c}: {_cap(desc)}"
    CODES_WITHOUT_DESCRIPTION[('procedure', c)] += 1
    return f"Procedure {c}"


def describe_mods(mod_part):
    mods = [m.strip().upper() for m in str(mod_part).split(',') if m.strip() not in ('-1', '0', '')]
    out = []
    for m in mods:
        desc = modifier_lookup.get(m, '')
        if not desc:
            CODES_WITHOUT_DESCRIPTION[('modifier', m)] += 1
        out.append(f"{m}={_cap(desc)}" if desc else m)
    return f"Modifiers: {', '.join(out)}" if out else ''


def decode_feature_codes(feature):
    """
    Decode raw codes embedded in feature names using reference lookup tables.

    Handles these patterns:
      procmodif_proc_pair_RR,N1,-1,-1__E1390  -> modifiers + procedure code
      proc_proc_mod_other_RR,-1__E1390        -> modifiers + procedure code (rare pairs)
      diag_cd_1_proc_cd_pair_Z89512__L5620    -> diagnosis + procedure code
      diag_proc_other_Z89512__L5620           -> diagnosis + procedure code (rare pairs)
      PROC_CD_E1390                           -> standalone procedure code
      PROC_MOD_1_RR                           -> standalone modifier

    Returns a decoded string or empty string if no codes found.
    """
    if pd.isna(feature):
        return ""
    feature_str = str(feature)
    decoded_parts = []
    try:
        for prefix in ('procmodif_proc_pair_', 'proc_proc_mod_other_'):
            if prefix in feature_str:
                suffix = feature_str.split(prefix, 1)[1]
                if '__' in suffix:
                    mod_part, proc_part = suffix.split('__', 1)
                    decoded_parts += [describe_proc(proc_part), describe_mods(mod_part)]
                break
        else:
            if ('diag_cd_' in feature_str and 'proc_cd_pair_' in feature_str) or 'diag_proc_other_' in feature_str:
                prefix = 'proc_cd_pair_' if 'proc_cd_pair_' in feature_str else 'diag_proc_other_'
                suffix = feature_str.split(prefix, 1)[1]
                if '__' in suffix:
                    diag_part, proc_part = suffix.split('__', 1)
                    decoded_parts += [describe_diag(diag_part), describe_proc(proc_part)]
            elif feature_str.startswith('PROC_CD_'):
                d = describe_proc(feature_str[len('PROC_CD_'):])
                if ':' in d:                       # the bare code is already in the label
                    decoded_parts.append(d)
            elif feature_str.startswith('PROC_MOD_'):
                parts = feature_str.split('_')
                mod = parts[-1].strip().upper() if len(parts) >= 4 else ''
                if mod and mod not in ('-1', '0'):
                    desc = modifier_lookup.get(mod, '')
                    if desc:
                        decoded_parts.append(f"Modifier {mod}: {_cap(desc)}")
                    else:
                        CODES_WITHOUT_DESCRIPTION[('modifier', mod)] += 1
    except (IndexError, ValueError):
        pass
    return " | ".join(p for p in decoded_parts if p)


# ── Quality checks ─────────────────────────────────────────────────────────────────────────────
# Flags starting with 'info:' are informational and do not fail QA.
QA_BANNED = [
    (r'\bscor(e|es|ed|ing)\b', 'score'), (r'\bz-?scores?\b', 'z-score'), (r'\banomal', 'anomaly'),
    (r'\boutliers?\b', 'outlier'), (r'\bmodel(s|ed|ing)?\b', 'model'), (r'\bshap\b', 'shap'),
    (r'\balgorithm', 'algorithm'), (r'machine learning|autoencoder|isolation forest', 'ml term'),
    (r'insanely', 'raw tier label'), (r'\bbilled\b', 'billed'), (r'\bfraud', 'fraud'),
    (r'\b(markedly|significantly|dramatically|substantially|extremely|grossly|alarmingly|egregious(ly)?)\b', 'intensifier'),
    (r'\b(deliberate(ly)?|intentional(ly)?|strategic(ally)?|exploit(s|ed|ing)?|scheme to|knowingly|purpose(ly|fully)'
     r'|to maximi[sz]e|in order to (increase|inflate|obtain|maximi[sz]e)|gam(e|ed|ing) the)\b', 'intent'),
    (r'medical(ly)? necess|clinically (justif|appropriate|necessary|indicated)|\bunjustified\b|not justified'
     r'|\bunnecessary\b|\b(did not|didn\'t) need\b|\bnot needed\b|\bno (medical )?need\b', 'medical necessity'),
    (r'\b(was|were|been|is|are) (not |never )?(actually )?(delivered|provided|rendered|performed)\b'
     r'|\bnot (actually )?(delivered|provided|rendered|received|performed)\b|\bnever (happened|occurred|took place|received)\b'
     r'|\bdid not (happen|occur|take place|receive)\b|\bmay (not|never) have (occurred|happened|received|been)\b'
     r'|\b(fake|fictitious|phony|bogus)\b', 'delivery claim'),
    (r'recommend|should (be )?investigat|\bwarrants?\b|further (review|investigation)|next step', 'recommendation'),
]
# Informal names that point to a scheme label; checked after exact labels are removed
SCHEME_SYNONYMS = [
    (r'double[- ]?bill|billed twice|paid twice|duplicate (claims|payments)', 'duplicate billing'),
    (r'\bup-?\s?cod(e|ed|ing)\b', 'upcoding'), (r'\bphantom\b', 'phantom billing'), (r'\bunbundl', 'unbundling'),
    (r'inflated (mileage|miles)|padded (mileage|miles|wait)', 'mileage inflation'),
    (r'\bkickbacks?\b', 'kickback'), (r'\bself-?referr', 'self-referral'),
]
_NUM_RE = re.compile(r'(?<![A-Za-z0-9_.\-])\$?\d[\d,]*(?:\.\d+)?(?:%|st|nd|rd|th)?(?![A-Za-z0-9_])')
_WORD_NUMBERS = r'(?:two|three|four|five|six|seven|eight|nine|ten)'
_RATIO_DIGIT_RE = re.compile(r'\b(\d+(?:\.\d+)?)\s*(?:times|-fold|fold)\s+(?:the|as|higher|more|greater|larger|that|what|peers?|its|their|than)\b'
                             r'|\b(\d+(?:\.\d+)?)\s*x\b')
_RATIO_WORD_RE = re.compile(rf'\b{_WORD_NUMBERS}\s*(?:times|-fold|fold)\s+(?:the|as|higher|more|greater|larger|that|what|peers?|its|their|than)\b'
                            r'|\btwice (?:as|the|that)\b|\b(?:double|triple|quadruple) (?:the|that)\b'
                            r'|\b(?:half|a third|one third|a quarter|one quarter|two thirds|three quarters) (?:of|as)\b'
                            rf'|\bone in (?:{_WORD_NUMBERS}|\d+)\b')
_UNIT_GROUPS = [{'patient', 'member', 'rider', 'beneficiary', 'resident', 'person', 'people'}, {'trip', 'ride'},
                {'claim'}, {'line', 'item'}, {'visit', 'encounter', 'appointment'}, {'day'}, {'holiday'}, {'mile'},
                {'unit'}, {'hour'}, {'minute'}, {'provider', 'peer', 'facility'}, {'code'}, {'week'}, {'month'},
                {'diagnosis', 'diagnose'}, {'modifier'}, {'combination', 'pair'}]
_SPELLED = {'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
            'eleven': 11, 'twelve': 12, 'fifteen': 15, 'twenty': 20}
_UNIT_SKIP = {'paid', 'more', 'fewer', 'additional', 'distinct', 'unique', 'different', 'total', 'separate',
              'such', 'of', 'the', 'its', 'their', 'per', 'for', 'and', 'or', 'in', 'on'}


def _sing(w):
    w = w.lower().strip(".,;:()'\"")
    return w[:-1] if w.endswith('s') and not w.endswith('ss') else w


def _unit_group(word):
    w = _sing(word)
    for gi, g in enumerate(_UNIT_GROUPS):
        if w in g:
            return gi
    return None


def _numbers(text):
    """[(token, value, kind, unit_group)] with kind in {'$', '%', 'ord', 'n'}; unit_group = noun right after."""
    t = normalize_text(text)
    out = []
    for m in _NUM_RE.finditer(t):
        tok = m.group(0)
        core = re.sub(r'[^\d.]', '', tok.replace(',', '')).rstrip('.')
        try:
            val = float(core)
        except ValueError:
            continue
        after = t[m.end():m.end() + 40].split()
        if tok.startswith('$'):
            kind = '$'
        elif tok.endswith('%') or (after and after[0].lower().startswith('percent') and not after[0].lower().startswith('percentile')):
            kind = '%'
        elif re.search(r'(st|nd|rd|th)$', tok):
            kind = 'ord'
        else:
            kind = 'n'
        unit = None
        for w in after[:3]:
            if _sing(w) in _UNIT_SKIP:
                continue
            unit = _unit_group(w)
            break
        out.append((tok, val, kind, unit, m.start()))
    return out


def _same_value(v, p):
    return v == p or (p >= 10 and v == round(p)) or (abs(p) >= 100 and abs(v - p) <= 0.01 * abs(p))


def quality_check(summary, expected_category, has_baselines=True, provider_id=None,
                  prompt_text=None, allowed_schemes=None, value_text=None):
    """
    Heuristic QA on a generated note. Returns a list of issues, or ['pass'].
    prompt_text: the user message the LLM saw (numbers in the note must appear there, same kind: $, %, ordinal).
    value_text: the Python-computed figures (peer comparison + value sentences); ratios and multiples must
                appear there, and a provider value must not be presented as the peer median (or the reverse).
    """
    issues = []
    s = normalize_text(summary)
    low = s.lower()
    n_words = len(s.split())
    if n_words > 250:
        issues.append("too_long")
    if n_words < 40:
        issues.append("too_short")

    # Scheme names: exact labels first, then informal names; then remove them so the word checks don't trip
    scrubbed = low
    for label in sorted([k for k in KNOWN_SCHEMES if k != 'other'], key=len, reverse=True):
        if label in scrubbed:
            if allowed_schemes is not None and label not in allowed_schemes:
                issues.append(f"unsupported_scheme:{label}")
            scrubbed = scrubbed.replace(label, ' ')
    for pat, label in SCHEME_SYNONYMS:
        m = re.search(pat, scrubbed)
        if m:
            if allowed_schemes is None or label not in allowed_schemes:
                issues.append(f"unsupported_scheme:{label}({m.group(0)})")
            scrubbed = re.sub(pat, ' ', scrubbed)
    for pat, name in QA_BANNED:
        m = re.search(pat, scrubbed)
        if m:
            issues.append(f"banned:{name}({m.group(0)})")

    sents = [x.strip() for x in _SENT.split(s.strip()) if x.strip()]
    first = sents[0] if sents else ''
    if provider_id is not None and not re.match(
            rf'\s*Provider\s+{re.escape(str(provider_id))}\s+was flagged because its\b', first):
        issues.append("opening_template")
    if not category_mentioned(first, expected_category):
        issues.append("category_not_in_opening")
    low_sents = [x.lower() for x in sents]
    if len(low_sents) != len(set(low_sents)):
        issues.append("repeated_sentences")
    if s.strip() and not s.rstrip().rstrip('"\')').endswith(('.', '!', '?')):
        issues.append("truncated_mid_sentence")

    if prompt_text:
        p_text = re.sub(r'(?m)^\s*\d+\.\s', ' ', normalize_text(prompt_text))   # list numbering is not data
        p_lines = p_text.splitlines()
        p_items = []
        for li, line in enumerate(p_lines):
            words = {_sing(w) for w in re.findall(r'[A-Za-z]+', line)}
            for tok, v, kind, unit, _ in _numbers(line):
                p_items.append((v, kind, words))
        ungrounded, unit_bad = set(), set()
        for tok, v, kind, unit, _ in _numbers(s):
            same = [w for (p, pk, w) in p_items if pk == kind and _same_value(v, p)]
            if not same:
                ungrounded.add(tok)
            elif unit is not None and not any(_UNIT_GROUPS[unit] & w for w in same):
                unit_bad.add(f"{tok} {_sing(s[s.find(tok) + len(tok):].split()[0]) if s[s.find(tok) + len(tok):].split() else ''}")
        # Spelled-out counts ("six members") must also exist, as digits, on a prompt line about the same thing
        for m in re.finditer(r'\b(three|four|five|six|seven|eight|nine|ten|eleven|twelve|fifteen|twenty)\s+'
                             r'([a-z]+)(?:\s+([a-z]+))?', low):
            w1, w2 = m.group(2), m.group(3) or ''
            unit = _unit_group(w1) if _sing(w1) not in _UNIT_SKIP else _unit_group(w2)
            if unit is None:
                continue
            v = float(_SPELLED[m.group(1)])
            if not any(pk == 'n' and _same_value(v, p) and (_UNIT_GROUPS[unit] & w) for (p, pk, w) in p_items):
                ungrounded.add(m.group(0))
        if ungrounded:
            issues.append(f"ungrounded_number:{'|'.join(sorted(ungrounded)[:5])}")
        if unit_bad:
            issues.append(f"number_label_mismatch:{'|'.join(sorted(unit_bad)[:3])}")

    vt = normalize_text(value_text or '')
    vt_vals = [v for _, v, _, _, _ in _numbers(vt)]
    bad_ratio = [m.group(0) for m in _RATIO_WORD_RE.finditer(low)]
    for m in _RATIO_DIGIT_RE.finditer(low):
        n = float(m.group(1) or m.group(2))
        if not (re.search(r'times', m.group(0)) and any(_same_value(n, p) for p in vt_vals)):
            bad_ratio.append(m.group(0))
    if bad_ratio:
        issues.append(f"invented_ratio:{'|'.join(bad_ratio[:3])}")

    # A provider value presented as the peer median, or a peer median presented as the provider's value
    pairs = []
    for line in vt.splitlines():
        pm = re.search(r'peer median (\$?[\d,]+(?:\.\d+)?%?)', line)
        if not pm:
            continue
        before = [tok for tok, _, kind, _, _ in _numbers(line[:pm.start()]) if kind != 'ord']
        if before:
            pairs.append((before[-1], pm.group(1)))
    for pv, pmed in pairs:
        if pv == pmed:
            continue
        if re.search(rf'median\s+(?:of\s+)?(?:about\s+|around\s+)?{re.escape(pv)}(?![\d,])', s):
            issues.append(f"swapped_value:{pv} called the peer median")
        if re.search(rf'\b(?:provider|it)\s+(?:was paid|averaged|had|received|reached)\s+(?:about\s+|around\s+)?{re.escape(pmed)}(?![\d,])', s, re.I):
            issues.append(f"swapped_value:{pmed} called the provider's value")

    if not has_baselines:
        issues.append("info:no_baseline_available")
    return issues if issues else ["pass"]


def qa_status(flags):
    real = [f for f in flags if f != 'pass' and not str(f).startswith('info:')]
    return 'pass' if not real else 'needs_review'


##### Provider Profile & Peer Comparison
Computed from `df_scoring_master` (all providers in the claim category, not just flagged). The prompt shows the provider's own figure next to the peer median and percentile, so the LLM never has to invent a comparison.

In [ ]:
def _ordinal(n):
    n = int(round(float(n)))
    suf = 'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f"{n}{suf}"


def _pct_ordinal(p):
    """Percentile as an ordinal, kept between 1st and 99th (100th and 0th read as errors)."""
    return _ordinal(min(max(float(p), 1), 99))


def _percentile(values, v):
    """Percent of peers below v (ties count half). values: pd.Series of peer values incl. v."""
    vals = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if vals.empty or v is None or pd.isna(v):
        return np.nan
    return float(((vals < v).mean() + 0.5 * (vals == v).mean()) * 100)


def _provider_agg(grp):
    """Per-provider paid-claims totals for one claim category."""
    line_stats = grp.groupby(provider_id).agg(avg_paid_per_line=('PMT_AMT', 'mean'))
    claims_stats = (grp.groupby([provider_id, 'claim_id'])['PMT_AMT'].sum().reset_index()
                    .groupby(provider_id)['PMT_AMT'].agg(avg_paid_per_claim='mean'))
    agg = grp.groupby(provider_id).agg(
        total_paid=('PMT_AMT', 'sum'),
        unique_patients=('MEMBER_KEY', 'nunique'),
        unique_proc_codes=('PROC_CD', 'nunique'),
        total_claims=('claim_id', 'nunique'),
        total_lines=('UNIQUE_KEY', 'nunique'),
    )
    agg['claims_per_patient'] = agg['total_claims'] / agg['unique_patients'].replace(0, np.nan)
    agg['line_items_per_claim'] = agg['total_lines'] / agg['total_claims'].replace(0, np.nan)
    return agg.join(line_stats).join(claims_stats)


def compute_cohort_baselines(df_scoring_master):
    """Per-claim-category peer medians (dict keyed by Category) and per-provider figures (cohort_provider_stats)."""
    global cohort_provider_stats
    baselines, cohort_provider_stats = {}, {}
    for cat, grp in df_scoring_master.groupby('Category'):
        agg = _provider_agg(grp)
        cohort_provider_stats[cat] = agg
        baselines[cat] = {f'median_{c}': float(agg[c].median()) for c in agg.columns}
        baselines[cat]['provider_count'] = int(len(agg))
    return baselines


def compute_proc_category_baselines(df_scoring_master):
    """Per-(Category, proc_cd_category) medians: {Category: {proc_cd_category: {stats}}}."""
    baselines = {}
    for (cat, pc), grp in df_scoring_master.groupby(['Category', 'proc_cd_category']):
        provider_agg = grp.groupby(provider_id).agg(
            total_quantity=('QUANTITY_PAID', 'sum'),
            total_line_items=('PROC_CD', 'count'),
        )
        if len(provider_agg) < 5:
            continue
        baselines.setdefault(cat, {})[pc] = {
            'median_quantity': float(provider_agg['total_quantity'].median()),
            'median_line_items': float(provider_agg['total_line_items'].median()),
            'provider_count': int(len(provider_agg)),
            'by_provider': provider_agg,
        }
    return baselines


def build_provider_profile(pid, category, df_scoring_master):
    """Provider's main services, main diagnoses and size, from its paid claim lines in this category."""
    provider_claims = df_scoring_master[
        (df_scoring_master[provider_id] == pid) & (df_scoring_master['Category'] == category)
    ]
    if provider_claims.empty:
        return None
    n_lines = len(provider_claims)
    top_proc_cats_str = ", ".join(
        f"{c} ({n / n_lines * 100:.0f}% of lines)"
        for c, n in provider_claims['proc_cd_category'].value_counts().head(3).items()
    )
    if 'diag_1_level_I' in provider_claims.columns:
        top_diag_str = ", ".join(
            f"{d} ({n / n_lines * 100:.0f}% of lines)"
            for d, n in provider_claims['diag_1_level_I'].dropna().value_counts().head(3).items()
        )
    else:
        top_diag_str = "Not available"
    cohort = df_scoring_master[df_scoring_master['Category'] == category]
    provider_lines = cohort.groupby(provider_id).size()
    return {
        'top_proc_categories': top_proc_cats_str,
        'top_diagnoses': top_diag_str or "Not available",
        'line_count': int(n_lines),
        'claim_count': int(provider_claims['claim_id'].nunique()),
        'line_pct_rank': _percentile(provider_lines, n_lines),
    }


_BASELINE_METRICS = [
    ('total_paid', 'Total paid', 'dollars'),
    ('total_claims', 'Paid claims', 'count'),
    ('unique_patients', 'Patients with paid claims', 'count'),
    ('avg_paid_per_claim', 'Average paid per claim', 'dollars'),
    ('claims_per_patient', 'Paid claims per patient', 'ratio'),
    ('line_items_per_claim', 'Paid lines per claim', 'ratio'),
]


def _fmt_metric(x, kind):
    if x is None or pd.isna(x):
        return 'n/a'
    if kind == 'dollars':
        return f"${x:,.0f}" if abs(x) >= 100 else f"${x:,.2f}"
    if kind == 'count':
        return f"{x:,.0f}"
    return f"{x:,.1f}"


def format_baselines_for_prompt(category, provider_features_df, cohort_baselines, proc_cat_baselines,
                                pid=None):
    """Provider's own figures next to the peer median and percentile, same claim category."""
    cb = cohort_baselines.get(category, {})
    if not cb:
        return "Baselines not available."
    stats = cohort_provider_stats.get(category)
    have_provider = stats is not None and pid in stats.index
    lines = [f"Compared with {cb.get('provider_count', '?')} {category} providers (paid claims, same period):"]
    for col_name, label, kind in _BASELINE_METRICS:
        med = cb.get(f'median_{col_name}')
        if have_provider:
            v = stats.at[pid, col_name]
            pct = _percentile(stats[col_name], v)
            lines.append(f"  {label}: {_fmt_metric(v, kind)} for this provider; peer median {_fmt_metric(med, kind)}"
                         + (f"; {_pct_ordinal(pct)} percentile" if not pd.isna(pct) else ""))
        else:
            lines.append(f"  {label}: peer median {_fmt_metric(med, kind)} (this provider's figure not available)")

    pc_baselines = proc_cat_baselines.get(category, {})
    if pc_baselines and provider_features_df is not None and len(provider_features_df):
        relevant = {str(f).replace('proc_cd_category_', '') for f in provider_features_df['Feature']
                    if str(f).startswith('proc_cd_category_')}
        matching = [(pc, st) for pc, st in pc_baselines.items() if pc in relevant][:5]
        if matching:
            lines.append("By procedure group (same claim category):")
            for pc, st in matching:
                bp = st['by_provider']
                if pid in bp.index:
                    q = bp.at[pid, 'total_quantity']
                    lines.append(f"  {pc}: {q:,.0f} paid units for this provider; peer median {st['median_quantity']:,.0f}; "
                                 f"{_pct_ordinal(_percentile(bp['total_quantity'], q))} percentile ({st['provider_count']} providers)")
                else:
                    lines.append(f"  {pc}: peer median {st['median_quantity']:,.0f} paid units ({st['provider_count']} providers)")
    return "\n".join(lines)


##### Structured Context Builder
MAIN PATTERNS = top 3 driver features by Impact_Score; SUPPORTING = next drivers; BACKGROUND = context-only features. No scores or importance-file text reach the prompt.

In [ ]:
def shown_indices(scores, narrate_roles, top_num=TOP_NUM):
    """Single source of truth for what the LLM sees: (driver indices strongest first, capped at top_num;
    background indices, capped at MAX_CONTEXT_ITEMS). Also used for schemes and QA grounding."""
    def _s(i):
        try:
            return float(scores[i])
        except (TypeError, ValueError, IndexError):
            return 0.0
    order = sorted(range(len(narrate_roles)), key=lambda i: -_s(i))
    drivers = [i for i in order if narrate_roles[i] == 'Driver'][:top_num]
    background = [i for i in order if narrate_roles[i] == 'Context only'][:MAX_CONTEXT_ITEMS]
    return drivers, background


def build_structured_context(features, purposes, scores, clinical_contexts=None,
                            feature_categories=None, feature_detailed=None,
                            importance_tiers=None, thresholds_list=None,
                            decoded_codes=None, top_num=TOP_NUM,
                            narrate_roles=None, brief_contexts=None):
    """
    Build the pattern blocks the LLM sees, from one provider's filtered, de-duplicated features.

    v3.5:
    - Order is by Impact_Score inside this function (no reliance on input order).
    - MAIN PATTERNS = top TIER1_DRIVERS Driver features; SUPPORTING = next Drivers up to top_num;
      BACKGROUND = top MAX_CONTEXT_ITEMS Context-only features. Upstream Importance_Tier is not used,
      because it was set before filtering.
    - No scores, Feature_Category or Feature_Detailed text is written into the prompt.
    - score_summary describes the shape of the driver scores in words, with no numbers.

    Returns (most_impactful_feature, most_impactful_purpose, most_impactful_context,
             score_summary, tier1_text, tier2_text, driver_count, context_text)
    """
    n = len(features)

    def _at(lst, i, default=''):
        return lst[i] if (lst is not None and i < len(lst) and lst[i] is not None) else default

    def _score(i):
        try:
            return float(scores[i])
        except (TypeError, ValueError, IndexError):
            return 0.0

    roles = [str(_at(narrate_roles, i, '')) for i in range(n)]
    drivers, background = shown_indices(scores, roles, top_num)

    def _line(k, i, brief=False):
        line = f"  {k}. {features[i]}"
        decoded = str(_at(decoded_codes, i, '')).strip()
        if decoded:
            line += f" [{decoded}]"
        desc = str(_at(brief_contexts, i, '') if brief else _at(clinical_contexts, i, '')).strip()
        if not desc:
            desc = str(_at(purposes, i, '')).strip()
        return line + (f": {desc}" if desc else "")

    context_text = "\n".join(_line(k + 1, i, brief=True) for k, i in enumerate(background)) or "None."
    if not drivers:
        return (None, '', '', '', '', '', 0, context_text)

    top_idx = drivers[0]
    d_scores = [_score(i) for i in drivers]
    if len(d_scores) == 1:
        score_summary = "One main pattern was found."
    elif d_scores[0] >= 2 * float(np.median(d_scores[1:])):
        score_summary = "One pattern stands out clearly from the others."
    else:
        score_summary = "Several patterns contributed at similar levels."

    tier1 = drivers[:TIER1_DRIVERS]
    tier2 = drivers[TIER1_DRIVERS:]
    tier1_text = "\n".join(_line(k + 1, i) for k, i in enumerate(tier1))
    tier2_text = "\n".join(_line(k + 1, i, brief=True) for k, i in enumerate(tier2)) or "None."
    return (features[top_idx], str(_at(purposes, top_idx, '')), str(_at(clinical_contexts, top_idx, '')),
            score_summary, tier1_text, tier2_text, len(drivers), context_text)


##### Prompt Builder

In [ ]:
def build_prompt(provider_id, category, most_impactful_feature,
                 most_impactful_purpose, most_impactful_context,
                 tier1_text, tier2_text, score_summary,
                 provider_profile=None, baselines_text=None,
                 context_text="None.", scheme_candidates=None):
    """
    Messages for GPT-OSS. v3.5: every rule the QA checks is stated here; scheme names are limited to the
    provider's dictionary-linked schemes; no task line asks for numbers or reasons the input does not contain.
    """
    scheme_candidates = [s for s in (scheme_candidates or []) if s in SCHEME_DEFINITIONS_BY_LABEL]
    if scheme_candidates:
        scheme_rule = ("- You may say a pattern is 'consistent with' one of these schemes, and only these: "
                       + "; ".join(scheme_candidates)
                       + ". Never say a scheme happened or was intended.\n")
        scheme_defs = "SCHEME DEFINITIONS:\n" + "\n".join(
            f"- {s}: {SCHEME_DEFINITIONS_BY_LABEL[s]}" for s in scheme_candidates) + "\n"
    else:
        scheme_rule = "- Do not name any billing scheme; describe the patterns only.\n"
        scheme_defs = ""

    profile_section = ""
    if provider_profile:
        profile_section = (
            "PROVIDER PROFILE (this claim category):\n"
            f"  Main services: {provider_profile.get('top_proc_categories', 'Unknown')}\n"
            f"  Main primary diagnoses: {provider_profile.get('top_diagnoses', 'Unknown')}\n"
            f"  Paid claim lines: {provider_profile.get('line_count', 0):,} on "
            f"{provider_profile.get('claim_count', 0):,} paid claims"
            + (f" ({_pct_ordinal(provider_profile['line_pct_rank'])} percentile for paid lines among peers)"
               if not pd.isna(provider_profile.get('line_pct_rank', np.nan)) else "") + "\n"
        )
    baselines_section = ""
    if baselines_text and baselines_text != "Baselines not available.":
        baselines_section = f"PEER COMPARISON:\n{baselines_text}\n"

    system_msg = (
        "You are a Medicaid program integrity analyst writing a short case note for investigators. "
        f"Reasoning: {REASONING_LEVEL}\n\n"
        "RULES:\n"
        "- Use only facts and numbers that appear in the user message. Do not calculate new numbers, ratios, "
        "multiples or differences, and do not round a number into a different one.\n"
        "- All figures come from PAID claims. Write 'paid' or 'was paid for'. Paid is not proof that a service "
        "did or did not happen: never say a service was or was not delivered, provided or real.\n"
        "- Say the provider is higher or lower than peers only when a number in the message shows it (a provider "
        "value with a peer median or percentile). Otherwise say the pattern was unusual compared with peers.\n"
        f"- Describe at most {MAX_PATTERNS} patterns, taken from MAIN PATTERNS, strongest first. SUPPORTING "
        "PATTERNS may only add detail to those patterns.\n"
        "- BACKGROUND items may be mentioned once, introduced with 'In addition,', and never as a reason for the flag.\n"
        "- When an 'Other explanations' note plausibly applies, mention it in one short clause.\n"
        + scheme_rule +
        "- Do not judge medical necessity or quality of care.\n"
        "- No recommendations or next steps. No headings, lists or bold text.\n"
        "- Never use these words: score, model, anomaly, outlier, z-score, algorithm, fraud, billed.\n"
        "- No intensifiers such as markedly, significantly, dramatically or extremely.\n"
        "- 120 to 200 words in 2 or 3 short paragraphs.\n"
        f"- The first sentence must begin exactly: Provider {provider_id} was flagged because its {category} paid claims\n\n"
        + scheme_defs
    )

    user_msg = (
        "Write a case note for this provider.\n\n"
        f"PROVIDER: {provider_id} | CLAIM CATEGORY: {category}\n\n"
        f"{profile_section}\n"
        f"{baselines_section}\n"
        f"STRONGEST PATTERN: {most_impactful_feature} (item 1 under MAIN PATTERNS). {score_summary}\n\n"
        "MAIN PATTERNS (the reasons the provider stood out, strongest first):\n"
        f"{tier1_text}\n\n"
        "SUPPORTING PATTERNS (may add detail to the main patterns):\n"
        f"{tier2_text}\n\n"
        "BACKGROUND (never a reason for the flag):\n"
        f"{context_text}\n\n"
        "TASK:\n"
        f"- Begin: 'Provider {provider_id} was flagged because its {category} paid claims ...'\n"
        "- For each main pattern, say in plain words what was measured and how the provider compared with peers, "
        "using only numbers given above.\n"
        "- Add one plausible other explanation where an 'Other explanations' note applies.\n"
        + ("- Where a pattern fits an allowed scheme, say it is consistent with that scheme in one clause.\n"
           if scheme_candidates else "")
    )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]


##### Main Summary Generator
Narrative with one retry on a cut-off answer; evidence-only note when no driver survives or generation fails. Scheme labels are derived from the dictionary (`derive_schemes`), not from the narrative.

In [ ]:
def evidence_only_summary(provider_id, category, background_labels=None, reason='no_driver'):
    """Factual note used when no narrative is possible. Contains no model output."""
    text = (f"Provider {provider_id} was flagged because its {category} paid claims differed from peers. ")
    if reason == 'generation_failed':
        text += "A written case note could not be produced for this run; the measures behind the flag are on the evidence card."
    else:
        text += ("None of the measures behind the flag can be narrated yet: they are background measures or have "
                 "definitions awaiting confirmation. The measures are listed on the evidence card.")
    if background_labels:
        text += " Background measures: " + "; ".join(background_labels[:MAX_CONTEXT_ITEMS]) + "."
    return text


def generate_fraud_summary(provider_id, category, all_features, all_purposes,
                          all_scores, all_clinical_contexts=None,
                          feature_categories=None, feature_detailed=None,
                          importance_tiers=None, all_thresholds=None,
                          decoded_codes=None,
                          provider_profile=None, baselines_text=None,
                          top_num=TOP_NUM, narrate_roles=None, brief_contexts=None,
                          scheme_candidates=None):
    """
    Returns (summary_text, info).
    info: status ('narrative' | 'evidence_only' | 'generation_failed'), messages, raw_final,
          analysis_chars, attempts, new_tokens, hit_token_limit, parse.
    The hidden reasoning channel is never returned as the summary.
    """
    parsed = {}
    for name, val in [('all_features', all_features), ('all_purposes', all_purposes),
                      ('all_scores', all_scores), ('all_clinical_contexts', all_clinical_contexts),
                      ('decoded_codes', decoded_codes), ('narrate_roles', narrate_roles),
                      ('brief_contexts', brief_contexts)]:
        parsed[name] = safe_literal_eval(val) if isinstance(val, str) else (list(val) if val is not None else None)

    all_features = [str(f) for f in parsed['all_features']]
    all_purposes = [str(f) for f in (parsed['all_purposes'] or [])]

    (most_impactful_feature, most_impactful_purpose, most_impactful_context,
     score_summary, tier1_text, tier2_text, driver_count, context_text) = build_structured_context(
        features=all_features, purposes=all_purposes, scores=parsed['all_scores'],
        clinical_contexts=parsed['all_clinical_contexts'], decoded_codes=parsed['decoded_codes'],
        top_num=top_num, narrate_roles=parsed['narrate_roles'], brief_contexts=parsed['brief_contexts'])

    info = {'status': None, 'messages': None, 'raw_final': '', 'analysis_chars': 0, 'attempts': 0,
            'new_tokens': 0, 'hit_token_limit': False, 'parse': ''}
    roles = parsed['narrate_roles'] or []
    background_labels = [f for f, r in zip(all_features, roles) if r == 'Context only']

    if driver_count == 0:
        info['status'] = 'evidence_only'
        return evidence_only_summary(provider_id, category, background_labels), info

    messages = build_prompt(
        provider_id, category, most_impactful_feature,
        most_impactful_purpose, most_impactful_context,
        tier1_text, tier2_text, score_summary,
        provider_profile=provider_profile, baselines_text=baselines_text,
        context_text=context_text, scheme_candidates=scheme_candidates)
    info['messages'] = messages

    final_text, analysis_text, meta = "", "", {}
    for budget in (MAX_NEW_TOKENS, RETRY_MAX_NEW_TOKENS):
        final_text, analysis_text, meta = generate_with_harmony(messages, max_new_tokens=budget, return_meta=True)
        info['attempts'] += 1
        cut_off = meta.get('hit_token_limit') and not normalize_text(final_text).rstrip('"\')').endswith(('.', '!', '?'))
        if meta.get('final_found') and not cut_off:
            break

    info.update(raw_final=final_text, analysis_chars=len(analysis_text), new_tokens=meta.get('new_tokens', 0),
                hit_token_limit=bool(meta.get('hit_token_limit')), parse=meta.get('parse', ''))
    if not final_text:
        info['status'] = 'generation_failed'
        return evidence_only_summary(provider_id, category, background_labels, reason='generation_failed'), info

    info['status'] = 'narrative'
    return clean_llm_summary(final_text), info


def _on_concern_side(direction, pct):
    """True when the provider sits on the side of peers the dictionary calls concerning, or when unknown."""
    if pct is None or pd.isna(pct) or direction not in ('High', 'Low'):
        return True
    return pct >= SCHEME_PCT_HIGH if direction == 'High' else pct <= SCHEME_PCT_LOW


def derive_schemes(features, scores, scheme_links, narrate_roles, concept_groups=None, max_n=MAX_PATTERNS,
                   peer_pcts=None, directions=None):
    """
    Scheme labels from the dictionary (Scheme_Link) of the driver features the LLM is shown (shown_indices),
    never from the narrative and never from drivers outside the prompt.
    - A driver counts only if the provider is on the concerning side of peers (Concern_Direction + percentile).
    - Evidence = number of distinct concept groups with a MEASURED percentile on the concerning side
      (3+ high, 2 medium, 1 low); drivers with no measured value can add a label but never raise evidence above low.
    - in_main: the label is linked to one of the MAIN PATTERNS; only those may be named in the narrative.
    - No linked driver -> 'other' (possible new pattern); no driver at all -> 'other' with evidence 'none'.
    """
    roles = [str(r) for r in narrate_roles]
    drivers, _ = shown_indices(scores, roles)
    main = set(drivers[:TIER1_DRIVERS])
    total, measured, groups, labels, in_main = (defaultdict(float), defaultdict(set), defaultdict(set),
                                                defaultdict(list), defaultdict(bool))
    for i in drivers:
        link = scheme_links[i] if i < len(scheme_links) else ''
        if not link or str(link).strip().lower() == 'none':
            continue
        pct = peer_pcts[i] if peer_pcts is not None and i < len(peer_pcts) else None
        direction = directions[i] if directions is not None and i < len(directions) else ''
        if not _on_concern_side(direction, pct):
            continue
        grp = concept_groups[i] if concept_groups and i < len(concept_groups) and concept_groups[i] else features[i]
        for lab in str(link).split('|'):
            lab = lab.strip().lower()
            if lab not in KNOWN_SCHEMES or lab == 'other':
                continue
            total[lab] += float(scores[i])
            in_main[lab] |= i in main
            if grp not in groups[lab]:
                groups[lab].add(grp)
                labels[lab].append(str(features[i]))
            if pct is not None and not pd.isna(pct):
                measured[lab].add(grp)
    if not total:
        if drivers:
            return [{'scheme': 'other', 'evidence': 'low', 'in_main': False,
                     'rationale': 'No shown driver measure links to a known scheme on the concerning side of peers; possible new pattern.'}]
        return [{'scheme': 'other', 'evidence': 'none', 'in_main': False,
                 'rationale': 'No narratable driver measure (on hold or background only).'}]
    ranked = sorted(total, key=lambda k: (-total[k], k))[:max_n]
    ev = lambda n: 'high' if n >= 3 else ('medium' if n == 2 else 'low')
    return [{'scheme': k, 'evidence': ev(len(measured[k])), 'in_main': in_main[k],
             'rationale': 'Linked driver measures: ' + '; '.join(labels[k][:4])
                          + ('' if measured[k] else ' (no provider value measured)')} for k in ranked]


def format_schemes_for_output(scheme_results):
    """Format scheme results for the output DataFrame columns."""
    display_parts, rationale_parts = [], []
    for r in scheme_results:
        scheme_name = SCHEME_DISPLAY_NAMES.get(r['scheme'], r['scheme'])
        if r['scheme'] == 'other' and r.get('evidence') == 'none':
            scheme_name = 'Not assessed (no narratable driver measure)'
        evidence_label = EVIDENCE_DISPLAY.get(r['evidence'], r['evidence'])
        display_parts.append(scheme_name if r.get('evidence') == 'none' else f"{scheme_name} ({evidence_label})")
        if r.get('rationale'):
            rationale_parts.append(f"{scheme_name}: {r['rationale']}")
    return ', '.join(display_parts), ' | '.join(rationale_parts)


##### Load Data

###### 1. Load scoring data

In [ ]:
scoring_type = 'rolling_6m_score'
scoring_type = 'rolling_3m_score'
SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/nemt_not_ihs/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
nemt_not_ihs_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
nemt_not_ihs_6m['Category'] = 'NEMT NOT IHS'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/nemt_ihs/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
nemt_ihs_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
nemt_ihs_6m['Category'] = 'NEMT IHS'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/dme_supplies/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
dme_supplies_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
dme_supplies_6m['Category'] = 'DME MEDICAL SUPPLIES'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/dme_and_appliances/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
dme_and_appliances_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
dme_and_appliances_6m['Category'] = 'DME AND APPLIANCES'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/nursing_home/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
nursing_home_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
nursing_home_6m['Category'] = 'NURSING HOME'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/bh_outpatient_clinic/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
bh_outpatient_clinic_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
bh_outpatient_clinic_6m['Category'] = 'BH OUTPATIENT CLINIC'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/bh_residential_treatment/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
bh_residential_treatment_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
bh_residential_treatment_6m['Category'] = 'BH RESIDENTIAL TREATMENT'

SCORING_DATA_ROOT = f'/Workspace/Users/{ROOT}/Projects/bh_professional_counseling/scoring_data'
scoring_df_path = f'{SCORING_DATA_ROOT}/{scoring_type}_monthly_scoring_file.csv.gz'
bh_professional_counseling_6m = pd.read_csv(scoring_df_path, compression='gzip', low_memory=False).reset_index(drop=True)
bh_professional_counseling_6m['Category'] = 'BH PROFESSIONAL'

In [ ]:
df_scoring_master = pd.concat([
    nemt_not_ihs_6m,
    nemt_ihs_6m,
    dme_supplies_6m,
    dme_and_appliances_6m,
    nursing_home_6m,
    bh_outpatient_clinic_6m,
    bh_residential_treatment_6m,
    bh_professional_counseling_6m
], ignore_index=True)

In [ ]:
df_scoring_master.rename(columns={'SProv_ID': provider_id}, inplace=True)
df_scoring_master[provider_id] = normalize_provider_id(df_scoring_master[provider_id])
_blank_ids = int(df_scoring_master[provider_id].isna().sum())
if _blank_ids:
    print(f"Dropping {_blank_ids:,} scoring lines with a blank provider ID")
df_scoring_master = df_scoring_master.dropna(subset=[provider_id]).reset_index(drop=True)
df_scoring_master['Category'] = df_scoring_master['Category'].astype(str).str.strip().str.upper()


In [ ]:
important_featues_all = pd.read_csv(
    f'/Workspace/Users/{ROOT}/Projects/all_projects_predictions/data/all_feature_importance_normalized_and_tiers.csv'
)[[provider_id, 'Feature', 'Impact_Score', 'Feature_Detailed', 'Feature_Category',
   'Importance_Tier', 'Scoring_Date', 'Category']]

In [ ]:
important_featues_all = important_featues_all[important_featues_all['Scoring_Date'] == scoring_date]

print(f"Providers to summarize: {important_featues_all[provider_id].nunique()}")
print(f"Rows (one per feature): {len(important_featues_all)}")
print(f"Columns: {list(important_featues_all.columns)}")

###### 2. Load external data

In [ ]:
icd10_desc = pd.read_csv(
    f'/Workspace/Users/{ROOT}/data/external data/icd10_categories_description.csv'
)
hcpcs_desc = pd.read_csv(
    f'/Workspace/Users/{ROOT}/data/external data/hcpcs_code_description.csv'
)

cpt_desc = pd.read_csv(
f'/Workspace/Users/{ROOT}/data/external data/cpt_codes_description.csv')

modifier_desc = pd.read_csv(
f'/Workspace/Users/{ROOT}/data/external data/procedure_modifiers.csv'
)

# ICD-10 groups by first 3 characters (same file the feature engineering uses for diag_N_level_I/II/III)
diag_hierarchy = pd.read_csv(
    f'/Workspace/Users/{ROOT}/data/external data/icd_10_diagnosis_code_root_and_category.csv', dtype=str
)


###### 3. Build code lookup dictionaries
Decode raw codes embedded in feature names into clinical descriptions using the reference tables.

In [ ]:
def build_code_lookups(icd10_desc, hcpcs_desc, cpt_desc, modifier_desc):
    """
    Build lookup dicts from reference tables. Handles varying column names.
    Returns: (proc_code_lookup, diag_code_lookup, modifier_lookup)
    """
    def _build_lookup(df, candidate_pairs):
        """Try candidate (code_col, desc_col) pairs, fallback to first two columns."""
        for code_col, desc_col in candidate_pairs:
            if code_col in df.columns and desc_col in df.columns:
                return {
                    str(row[code_col]).strip().upper(): str(row[desc_col]).strip()
                    for _, row in df.dropna(subset=[code_col, desc_col]).iterrows()
                }
        # Fallback: first two columns
        cols = df.columns.tolist()
        if len(cols) >= 2:
            return {
                str(row[cols[0]]).strip().upper(): str(row[cols[1]]).strip()
                for _, row in df.dropna(subset=[cols[0], cols[1]]).iterrows()
            }
        return {}

    # Procedure codes: HCPCS + CPT merged
    proc_code_lookup = _build_lookup(hcpcs_desc, [
        ('HCPCS Codes', 'Description'), ('hcpcs_code', 'description'),
        ('Code', 'Description'), ('HCPCS_CD', 'Description')
    ])
    cpt_lookup = _build_lookup(cpt_desc, [
        ('CPT Code', 'Description'), ('cpt_code', 'description'),
        ('Code', 'Description'), ('CPT_CD', 'Description')
    ])
    for k, v in cpt_lookup.items():
        if k not in proc_code_lookup:
            proc_code_lookup[k] = v

    # Diagnosis codes: ICD-10
    diag_code_lookup = _build_lookup(icd10_desc, [
        ('ICD10 Code', 'Description'), ('icd10_code', 'description'),
        ('Code', 'Description'), ('DIAG_CD', 'Description'),
        ('ICD-10-CM Code', 'Description')
    ])

    # Modifiers
    modifier_lookup = _build_lookup(modifier_desc, [
        ('Modifier', 'Description'), ('modifier', 'description'),
        ('Code', 'Description'), ('PROC_MOD', 'Description'),
        ('Modifier Code', 'Modifier Description')
    ])

    print(f"Lookup tables built:")
    print(f"  Procedure codes (HCPCS+CPT): {len(proc_code_lookup):,}")
    print(f"  Diagnosis codes (ICD-10):     {len(diag_code_lookup):,}")
    print(f"  Modifiers:                    {len(modifier_lookup):,}")
    return proc_code_lookup, diag_code_lookup, modifier_lookup

proc_code_lookup, diag_code_lookup, modifier_lookup = build_code_lookups(
    icd10_desc, hcpcs_desc, cpt_desc, modifier_desc
)

# ── ICD-10 group descriptions: used when the full diagnosis code is not in icd10_desc ──
_dh = diag_hierarchy.rename(columns=lambda c: c.strip()).rename(columns={'diag_cd_root': 'diag_root'})
_dh['diag_root'] = _dh['diag_root'].astype(str).str.strip().str.upper()
_dup = int(_dh['diag_root'].duplicated().sum())
if _dup:
    print(f"WARNING: {_dup} duplicate diagnosis roots in the ICD-10 group file (first kept). The same duplicates "
          "would multiply claim lines in the feature-engineering merge.")
_dh = _dh.drop_duplicates('diag_root')
diag_group_lookup = dict(zip(_dh['diag_root'], _dh['category_level_III'].fillna('').astype(str).str.strip()))
diag_group_lookup = {k: v for k, v in diag_group_lookup.items() if v}
# Full-code keys without dots, so 'F43.12' in the table matches 'F4312' in a feature name
diag_code_lookup = {k.replace('.', ''): v for k, v in diag_code_lookup.items()}
print(f"  ICD-10 groups (3-character):  {len(diag_group_lookup):,}")


##### Feature Dictionary (v2.1)
Loads `feature_mapping.csv` (export of AHCCCS_feature_dictionary_v2.1.xlsx) and checks it before use.

In [ ]:
# ── Feature dictionary v2.1 (exported to feature_mapping.csv; see Read Me sheet of the workbook) ──
MAPPING_CSV = f'/Workspace/Users/{ROOT}/Projects/all_projects_predictions/data/feature_mapping.csv'

if DICTIONARY_XLSX_PATH:   # regenerate the CSV from the workbook (Step 0 of the wiring)
    _d = pd.read_excel(DICTIONARY_XLSX_PATH, sheet_name='Dictionary').fillna('')
    _d = _d.rename(columns={'Display_Label': 'Replacement'})
    _d['Purpose'] = _d['Description']
    _d['Clinical_Context'] = _d['Description']
    _d['Thresholds'] = ''
    _d.to_csv(MAPPING_CSV, index=False)

mapping_df = pd.read_csv(MAPPING_CSV, dtype=str).fillna('')
REQUIRED_MAPPING_COLS = ['Feature_Root', 'Replacement', 'Description', 'Suspicious_Pattern', 'Benign_Explanations',
                         'Narrative_Caveat', 'Value_Template', 'Narrate_Role', 'Validation_Status', 'Applies_To',
                         'Concept_Group', 'Value_Type', 'Scheme_Link', 'Purpose', 'Clinical_Context', 'Thresholds']
_missing_cols = [c for c in REQUIRED_MAPPING_COLS if c not in mapping_df.columns]
assert not _missing_cols, f"feature_mapping.csv is not a v2.1 export; missing columns: {_missing_cols}"
for c in mapping_df.columns:
    mapping_df[c] = mapping_df[c].astype(str).str.strip()
if 'Concern_Direction' not in mapping_df.columns:
    print("WARNING: no Concern_Direction column; schemes will be linked without checking the provider's side of peers.")
    mapping_df['Concern_Direction'] = ''
_bad_dir = set(mapping_df['Concern_Direction']) - {'High', 'Low', 'Either', ''}
assert not _bad_dir, f"Unexpected Concern_Direction values: {_bad_dir}"
assert mapping_df['Feature_Root'].is_unique, "Duplicate Feature_Root rows in feature_mapping.csv"
assert (mapping_df['Description'] != '').all(), "Every dictionary row must have a Description"

_bad_links = {lab.strip() for v in mapping_df['Scheme_Link'] for lab in v.split('|')
              if lab.strip() and lab.strip().lower() != 'none' and lab.strip().lower() not in KNOWN_SCHEMES}
assert not _bad_links, f"Scheme_Link labels not in KNOWN_SCHEMES: {_bad_links}"
_bad_roles = set(mapping_df['Narrate_Role']) - {'Driver', 'Context only', 'Do not narrate'}
assert not _bad_roles, f"Unexpected Narrate_Role values: {_bad_roles}"

MAP = mapping_df.set_index('Feature_Root').to_dict('index')
ROOTS_LONGEST_FIRST = sorted(MAP, key=len, reverse=True)
ONE_HOT_VALUE_TYPES = {'Category level', 'Tier label', 'Yes/No flag', 'Date part'}
SUFFIX_OK_ROOTS = {r for r, v in MAP.items() if v['Value_Type'] in ONE_HOT_VALUE_TYPES}

# Dicts kept for the v3.4 helpers (sorted longest first, as before)
sort_key = lambda x: len(x[0])
mapping_dict = dict(sorted(zip(mapping_df['Feature_Root'], mapping_df['Replacement']), key=sort_key, reverse=True))
purpose_dict = dict(sorted(zip(mapping_df['Feature_Root'], mapping_df['Purpose']), key=sort_key, reverse=True))
clinical_context_dict = dict(sorted(zip(mapping_df['Feature_Root'], mapping_df['Clinical_Context']), key=sort_key, reverse=True))
thresholds_dict = {}

NEW_COLS = ['Suspicious_Pattern', 'Benign_Explanations', 'Narrative_Caveat', 'Narrate_Role',
            'Validation_Status', 'Applies_To', 'Concept_Group', 'Value_Template', 'Value_Type', 'Scheme_Link',
            'Concern_Direction']

# ── Tier and band labels arrive inside the feature name ('..._class_Rare Anomaly'); translate them ──
# classify_zscore() returns 'Normal' for missing values (too few lines, fewer than 3 providers, zero spread)
# and scores are capped at +/-10, so 'Rare Anomaly' is the top tier actually reached.
TIER_ROOTS = {'average_procedure_zscore_class', 'global_procedure_zscore_class', 'mileage_zscore_class',
              'not_mileage_zscore_class', 'procedure_volume_zscore_class', 'reservation_zscore_class',
              'wait_time_zscore_class'}
TIER_PLAIN = {
    'extreme low underuse': 'far below peers', 'very low underuse': 'well below peers',
    'slightly below average underuse': 'below peers', 'slightly below avg underuse': 'below peers',
    'normal': 'in line with peers, or too little data to compare',
    'mild anomaly': 'somewhat above peers', 'anomaly': 'well above peers',
    'rare anomaly': 'far above peers (highest level)', 'extreme anomaly': 'far above peers (highest level)',
    'insanely rare': 'far above peers (highest level)',
}
WAIT_BANDS = {  # unusual_wait_time_class: fixed hours, not peer comparisons (1 T2007 unit = 30 minutes)
    'normal': 'up to 2 hours', 'high': '2 to 4 hours', 'extremely high': '4 to 8 hours',
    'abnormal': '8 to 24 hours', 'suspicious': 'more than 24 hours', 'no wait time': 'no wait time',
}
# Levels that describe the provider as normal or below peers on an above-peers measure, or a yes/no flag
# that is absent. They can explain the model but are never a reason for the flag: narrated as background only.
NON_CONCERNING_LEVELS = {'in line with peers, or too little data to compare', 'below peers', 'well below peers',
                         'far below peers', 'not present', 'up to 2 hours', 'no wait time',
                         'below the top 5% of providers'}
RISKY_LEVEL_WORDS = re.compile(r'anomal|suspicious|rare|outlier|extreme|insane|abnormal|watch', re.I)
UNKNOWN_LEVELS = set()   # (root, raw label) pairs not translated; reported at the end of the run

print(f"Dictionary {DICTIONARY_VERSION}: {len(MAP)} roots | narrated: "
      f"{(mapping_df['Narrate_Role'] != 'Do not narrate').sum()} | on hold: "
      f"{(mapping_df['Validation_Status'] == 'Hold until confirmed').sum()} | one-hot roots: {len(SUFFIX_OK_ROOTS)}")


In [ ]:
# ── Provider values for Value_Template sentences (computed in Python, never by the LLM) ──
# A value is filled only when it can be computed without guessing what the model column means:
#   A. one-hot features (code, category, tier, yes/no) whose raw column is in the scoring file:
#      value = share of the provider's paid claim lines with that level
#   B. numeric features whose column is in the scoring file AND is constant within every provider
#      (a provider-level measure merged onto claim lines): value = that constant
# Everything else gets no sentence. Peers = all providers in the same claim category.
# Values outside the range their Value_Type allows (for example scaled or negative counts) are rejected.
NON_NEGATIVE_TYPES = {'Count', 'Days', 'Miles', 'Units', 'Dollars'}
UNIT_INTERVAL_TYPES = {'Share (0-1)', 'Index (0-1)'}
NEVER_QUOTE_TYPES = {'Standardized score', 'Date part'}


# Ratio measures that can legitimately be negative (growth rates; PMI-type association measures)
SIGNED_RATIO_ROOTS = {'billing_growth_rate', 'diag_pair_pmi_max', 'diag_pair_pmi_mean',
                      'proc_mod_pair_pmi_max', 'proc_mod_pair_pmi_mean'}


def _norm_level(x):
    """'1', 1, 1.0 and '1.0' compare equal; text levels are stripped."""
    s = str(x).strip()
    return re.sub(r'^(-?\d+)\.0+$', r'\1', s)


def _norm_level_series(col):
    return col.astype(str).str.strip().str.replace(r'^(-?\d+)\.0+$', r'\1', regex=True)


def _looks_scaled(root, vtype, per_provider):
    """True when values cannot be raw paid-claims figures (min-max or z-scaled columns)."""
    v = per_provider.dropna()
    if v.empty:
        return True
    if vtype in UNIT_INTERVAL_TYPES:
        return bool(((v < 0) | (v > 1)).any())
    if vtype in NON_NEGATIVE_TYPES:
        return bool((v < 0).any() or (v.between(0, 1).all() and v.nunique() > 2))
    if vtype == 'Ratio' and root not in SIGNED_RATIO_ROOTS:
        return bool((v < 0).any())
    return False


def build_provider_feature_values(df_scoring_master, needed):
    """
    needed: DataFrame with [provider_id, 'Category', 'Feature'] (features that will be narrated).
    Returns (values_df, coverage) where values_df has
    [provider_id, 'Category', 'Feature', 'value', 'peer_median', 'peer_pct', 'n_claims', 'value_source'].
    """
    rows, coverage = [], Counter()
    for cat, need_cat in needed.groupby('Category'):
        grp = df_scoring_master[df_scoring_master['Category'] == cat]
        if grp.empty:
            coverage['category_not_in_scoring_data'] += need_cat['Feature'].nunique()
            continue
        pids = grp[provider_id]
        n_claims = grp.groupby(provider_id)['claim_id'].nunique()
        for feat, need_f in need_cat.groupby('Feature'):
            root = match_root(feat)
            vtype = MAP.get(root, {}).get('Value_Type', '')
            level_raw = feat[len(root) + 1:] if feat != root else ''
            if vtype in NEVER_QUOTE_TYPES:
                coverage['not_quoted_by_design'] += 1
                continue
            if root not in grp.columns:
                coverage['column_not_in_scoring_data'] += 1
                continue
            col_vals = grp[root]
            measured_at = MAP.get(root, {}).get('Measured_At', '')
            if vtype == 'Yes/No flag' and measured_at == 'Provider':
                flag = pd.to_numeric(col_vals, errors='coerce')
                if (flag.groupby(pids).nunique(dropna=True) > 1).mean() > 0.01:
                    coverage['varies_within_provider'] += 1
                    continue
                per_provider = flag.groupby(pids).first().dropna()
                source = 'provider_flag'
            elif vtype in ONE_HOT_VALUE_TYPES:
                if measured_at != 'Claim line':
                    coverage['share_of_lines_would_mislabel'] += 1    # e.g. a claim- or trip-level measure
                    continue
                if vtype == 'Yes/No flag':
                    # The template always describes the 'yes' condition (e.g. holiday-dated lines), so the value is
                    # the share of lines where the flag is 1, also for a '_0' feature (whose label says 'not present').
                    ind = pd.to_numeric(col_vals, errors='coerce') == 1
                else:
                    norm_col = _norm_level_series(col_vals)
                    ind = norm_col == _norm_level(level_raw)
                    if not ind.any():
                        ind = norm_col.str.replace(' ', '_') == _norm_level(level_raw)
                if not ind.any():
                    coverage['level_not_found_in_column'] += 1
                    continue
                per_provider = ind.groupby(pids).mean()
                source = 'share_of_lines'
            else:
                num = pd.to_numeric(col_vals, errors='coerce')
                spread = num.groupby(pids).nunique(dropna=True)
                if (spread > 1).mean() > 0.01:
                    coverage['varies_within_provider'] += 1
                    continue
                per_provider = num.groupby(pids).first().dropna()
                if _looks_scaled(root, vtype, per_provider):
                    coverage['out_of_range_looks_scaled'] += 1
                    continue
                source = 'provider_measure'
            peer_median = float(per_provider.median())
            for pid in need_f[provider_id].unique():
                if pid not in per_provider.index:
                    coverage['provider_not_in_scoring_data'] += 1
                    continue
                v = float(per_provider.at[pid])
                rows.append({provider_id: pid, 'Category': cat, 'Feature': feat,
                             'value': ('yes' if v >= 1 else 'no') if source == 'provider_flag' else v,
                             'peer_median': peer_median, 'peer_pct': _percentile(per_provider, v),
                             'n_claims': int(n_claims.get(pid, 0)), 'value_source': source})
                coverage[source] += 1
    cols = [provider_id, 'Category', 'Feature', 'value', 'peer_median', 'peer_pct', 'n_claims', 'value_source']
    return pd.DataFrame(rows, columns=cols), coverage


def _fmt(x, vtype):
    try:
        if x is None or pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, str):
        return x
    x = float(x)
    if vtype == 'Dollars':
        return f"${x:,.0f}" if abs(x) >= 100 else f"${x:,.2f}"
    if vtype in ('Share (0-1)', 'Category level', 'Tier label', 'Yes/No flag'):
        if not 0 <= x <= 1:
            return f"{x:,.1f}"
        if 0.999 <= x < 1:          # avoid '100.0%' for a share that is not all
            x = 0.999
        elif 0 < x < 0.001:          # avoid '0.0%' for a share that is not none
            x = 0.001
        return f"{x:.1%}"
    if vtype in ('Count', 'Days', 'Miles', 'Units'):
        return f"{x:,.0f}" if float(x).is_integer() else f"{x:,.1f}"
    return f"{x:,.2f}"


def render_value_sentence(row):
    """Fill the dictionary Value_Template; return '' unless every placeholder can be filled honestly."""
    tpl = str(row.get('Value_Template', '') or '')
    if not tpl:
        return ''
    needed = set(re.findall(r'\{(\w+)\}', tpl))
    if needed & {'value', 'peer_median', 'peer_pct'}:          # a number will be quoted
        nc = row.get('n_claims')
        if nc is None or pd.isna(nc) or nc < MIN_CLAIMS_FOR_VALUE:
            return ''
    vtype = row.get('Value_Type', '')
    pct = row.get('peer_pct')
    vals = {
        'value': _fmt(row.get('value'), vtype),
        'peer_median': _fmt(row.get('peer_median'), vtype),
        'peer_pct': _pct_ordinal(pct) if pct is not None and not pd.isna(pct) else None,
        'level': row.get('Level_Plain', '') or None,
        'category': CATEGORY_DISPLAY.get(str(row['Category']).upper(), str(row['Category'])),
    }
    if any(vals.get(k) in (None, '') for k in needed):
        return ''
    out = tpl.format(**{k: vals[k] for k in needed})
    if vtype == 'Units' and 'wait' in str(row['Feature']).lower() and '{value}' in tpl:
        hrs = float(row['value']) * 0.5             # T2007: 1 unit = 30 minutes
        out = out.replace(f"{vals['value']} ", f"{vals['value']} (about {hrs:,.1f} hours) ", 1)
    return out


##### Prepare Features
Coverage gate (the run stops on any feature without a dictionary row), filters, de-duplication, Python-computed value sentences, then one row per provider and category.

In [ ]:
df = important_featues_all.copy()
df[provider_id] = normalize_provider_id(df[provider_id])
df['Feature'] = df['Feature'].astype(str).str.strip()
df['Category'] = df['Category'].astype(str).str.strip().str.upper()
df['Impact_Score'] = pd.to_numeric(df['Impact_Score'], errors='coerce')
prep_stats = Counter()
prep_stats['dropped_blank_provider_id'] = int(df[provider_id].isna().sum())
prep_stats['dropped_blank_score'] = int(df['Impact_Score'].isna().sum())
df = df.dropna(subset=[provider_id, 'Impact_Score'])
df = df[df['Feature'].str.lower() != 'index']          # row-index artifact, not a model feature
_n = len(df)
df = (df.sort_values('Impact_Score', ascending=False, kind='stable')
        .drop_duplicates(subset=[provider_id, 'Category', 'Feature'], keep='first'))
prep_stats['dropped_duplicate_feature_rows'] = _n - len(df)
all_provider_keys = df[[provider_id, 'Category']].drop_duplicates()
prep_stats['feature_rows'] = len(df)
prep_stats['providers'] = len(all_provider_keys)

# (0) Coverage gate: every feature must map to a dictionary row with a description.
df['Feature_Root'] = df['Feature'].apply(match_root)
_unmapped = sorted(df.loc[df['Feature_Root'] == '', 'Feature'].unique())
if _unmapped:
    _hints = []
    for f in _unmapped[:30]:
        near = next((r for r in ROOTS_LONGEST_FIRST if f.startswith(r)), '')
        _hints.append(f"  {f}" + (f"   (starts with root '{near}', Value_Type '{MAP[near]['Value_Type']}'; "
                                   f"suffixes are allowed only for one-hot roots)" if near else ""))
    raise ValueError(f"{len(_unmapped)} features have no dictionary row. Add them to the dictionary "
                     f"(or fix the root) and re-export feature_mapping.csv:\n" + "\n".join(_hints))

# (1) Dictionary fields, labels, purpose, decoded codes
for c in NEW_COLS + ['Description']:
    df[c] = df['Feature_Root'].map(lambda r, c=c: MAP[r].get(c, ''))
df['Level_Plain'] = df['Feature'].apply(plain_level)
df['Impactful_Feature'] = df['Feature'].apply(simplify_feature_name)
df['Impactful_Feature_Purpose'] = df['Description']
df['Feature_Thresholds'] = ''
df['Code_Decoded'] = df['Feature'].apply(decode_feature_codes)

# (2) Filters: not narrated, on hold, or not applicable to this claim category
def _applies(applies_to, category):
    a = str(applies_to).strip().upper()
    cat = str(category).strip().upper()
    if a in ('', 'ALL CATEGORIES'):
        return True
    if a.startswith('ALL EXCEPT'):
        return cat not in [x.strip() for x in a.replace('ALL EXCEPT', '').split(';')]
    return cat in [x.strip() for x in a.split(';')]

# A level that says the provider is normal, below peers, or the flag is absent cannot be a reason for the flag
_not_concerning = (df['Narrate_Role'] == 'Driver') & df['Level_Plain'].isin(NON_CONCERNING_LEVELS)
df.loc[_not_concerning, 'Narrate_Role'] = 'Context only'
prep_stats['drivers_moved_to_background'] = int(_not_concerning.sum())

_n = len(df)
df = df[df['Narrate_Role'].isin(['Driver', 'Context only'])]
prep_stats['dropped_not_narrated'] = _n - len(df); _n = len(df)
df = df[df['Validation_Status'] != 'Hold until confirmed']
prep_stats['dropped_on_hold'] = _n - len(df); _n = len(df)
df = df[[_applies(a, c) for a, c in zip(df['Applies_To'], df['Category'])]]
prep_stats['dropped_not_applicable'] = _n - len(df); _n = len(df)

# (3) One line per concept per provider. One-hot codes of the same field are different evidence,
#     so each code keeps its own line. Tier and yes/no levels are grouped by role, so a background level
#     ('in line with peers') never displaces a concerning level of the same measure.
#     Within a group the strongest feature is kept; a Driver outranks a stronger Context-only feature
#     only if it scored within DRIVER_PRIORITY_RATIO of it.
_onehot = df['Feature'] != df['Feature_Root']
_onehot_code = _onehot & (df['Value_Type'] == 'Category level')
_onehot_level = _onehot & df['Value_Type'].isin(['Tier label', 'Yes/No flag'])
df['_dedup_key'] = np.where(df['Concept_Group'] != '', df['Concept_Group'], df['Feature'])
df.loc[_onehot_code, '_dedup_key'] = df.loc[_onehot_code, '_dedup_key'] + '|' + df.loc[_onehot_code, 'Feature']
df.loc[_onehot_level, '_dedup_key'] = df.loc[_onehot_level, '_dedup_key'] + '|' + df.loc[_onehot_level, 'Narrate_Role']
_grp_keys = [provider_id, 'Category', '_dedup_key']
df['_grp_max'] = df.groupby(_grp_keys)['Impact_Score'].transform('max')
df['_driver_first'] = ((df['Narrate_Role'] == 'Driver')
                       & (df['Impact_Score'] >= DRIVER_PRIORITY_RATIO * df['_grp_max'])).astype(int)
df = df.sort_values([provider_id, 'Category', '_dedup_key', '_driver_first', 'Impact_Score'],
                    ascending=[True, True, True, False, False], kind='stable')
df['_is_kept'] = ~df.duplicated(subset=_grp_keys, keep='first')
_kept_score = df['Impact_Score'].where(df['_is_kept']).groupby([df[k] for k in _grp_keys]).transform('max')
df['Related_Measures'] = ((~df['_is_kept']) & (df['Impact_Score'] >= RELATED_MIN_RATIO * _kept_score)
                          ).groupby([df[k] for k in _grp_keys]).transform('sum').astype(int)
prep_stats['merged_near_duplicates'] = int((~df['_is_kept']).sum())
df = df[df['_is_kept']].drop(columns=['_dedup_key', '_grp_max', '_driver_first', '_is_kept'])

# (4) Provider value sentences (Python-computed; the LLM only copies them)
provider_feature_values, value_coverage = build_provider_feature_values(
    df_scoring_master, df[[provider_id, 'Category', 'Feature']].drop_duplicates())
assert not provider_feature_values.duplicated([provider_id, 'Category', 'Feature']).any()
df = df.merge(provider_feature_values, on=[provider_id, 'Category', 'Feature'], how='left', validate='m:1')
df['Value_Sentence'] = df.apply(render_value_sentence, axis=1)

# (5) LLM-facing text per feature. Scheme names are removed from the per-feature text: which schemes the
#     narrative may name is decided per provider (derive_schemes) and stated once in the system prompt.
_SCHEME_CLAUSE = re.compile(r'[,;]?\s*consistent with [^.;]*|\s*\(possible [^)]*\)', re.I)

def _compose(row):
    suspicious = _SCHEME_CLAUSE.sub('', row['Suspicious_Pattern']).strip()
    if suspicious and not suspicious.endswith(('.', '!', '?')):
        suspicious += '.'
    parts = [row['Description'],
             row['Value_Sentence'],
             suspicious,
             f"Other explanations: {row['Benign_Explanations']}" if row['Benign_Explanations'] else '',
             f"Note: {row['Narrative_Caveat']}" if row['Narrative_Caveat'] else '',
             f"({int(row['Related_Measures'])} closely related measure(s) also contributed.)"
             if row['Related_Measures'] > 0 else '']
    return ' '.join(str(p).strip() for p in parts if p and str(p).strip())

df['Clinical_Context'] = df.apply(_compose, axis=1)
df['Brief_Context'] = df.apply(lambda r: ' '.join(p for p in [r['Description'], r['Value_Sentence']] if p), axis=1)

# (6) Strongest first inside each provider, then one row per provider and category
df = df.sort_values([provider_id, 'Category', 'Impact_Score'], ascending=[True, True, False], kind='stable')
features_narrated = df.copy()

agg_dict = {
    'Impactful_Feature': list, 'Impactful_Feature_Purpose': list, 'Impact_Score': list,
    'Clinical_Context': list, 'Brief_Context': list, 'Feature_Thresholds': list, 'Code_Decoded': list,
    'Feature': list, 'Narrate_Role': list, 'Scheme_Link': list, 'Concept_Group': list,
    'Value_Sentence': list, 'peer_pct': list, 'Concern_Direction': list,
}
important_featues_all_described = (
    df.groupby([provider_id, 'Category']).agg(agg_dict).reset_index()
)
# Providers whose every feature was filtered out still get an (evidence-only) row
important_featues_all_described = all_provider_keys.merge(
    important_featues_all_described, on=[provider_id, 'Category'], how='left')
for c in agg_dict:
    important_featues_all_described[c] = important_featues_all_described[c].apply(
        lambda v: v if isinstance(v, list) else [])

prep_stats['feature_rows_narratable'] = len(df)
prep_stats['with_value_sentence'] = int((df['Value_Sentence'] != '').sum())
print("Feature preparation:", dict(prep_stats))
print("Value sentence sources:", dict(value_coverage))
print(f"Providers: {len(important_featues_all_described)} | with no narratable feature: "
      f"{int((important_featues_all_described['Feature'].str.len() == 0).sum())}")


##### Compute Peer Baselines

In [ ]:
# ── Peer figures from df_scoring_master (all providers, not just flagged), per claim category ──
cohort_baselines = compute_cohort_baselines(df_scoring_master)
print(f"  Cohort baselines for {len(cohort_baselines)} categories")

print("Computing procedure-category baselines...")
proc_cat_baselines = compute_proc_category_baselines(df_scoring_master)
total_pc = sum(len(v) for v in proc_cat_baselines.values())
print(f"  Procedure-category baselines: {total_pc} category-proc pairs")

_missing_cats = set(important_featues_all_described['Category']) - set(cohort_baselines)
if _missing_cats:
    print(f"WARNING: no scoring data for categories {_missing_cats}; their notes will have no peer comparison.")

for cat, stats in list(cohort_baselines.items())[:3]:
    print(f"\n  {cat}: {stats['provider_count']} providers, "
          f"median paid claims={stats['median_total_claims']:,.0f}, "
          f"median paid=${stats['median_total_paid']:,.0f}")


##### Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai_harmony import load_harmony_encoding, HarmonyEncodingName
import torch

# Token from the environment or a Databricks secret; never hard-coded in the notebook.
_hf_token = os.environ.get('HF_TOKEN')
if not _hf_token and HF_SECRET_SCOPE:
    _hf_token = dbutils.secrets.get(scope=HF_SECRET_SCOPE, key=HF_SECRET_KEY)
if _hf_token:
    login(_hf_token)
else:
    print("No Hugging Face token configured (HF_TOKEN or HF_SECRET_SCOPE); using cached model files.")

print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

# Load harmony encoder for proper channel parsing
harmony_enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

print(f"Model loaded: {MODEL_ID}")
print(f"Harmony encoder loaded: {HarmonyEncodingName.HARMONY_GPT_OSS}")
print(f"Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'auto'}")


##### Generate Summaries

In [ ]:
summary_rows, audit_rows = [], []
_n_total = len(important_featues_all_described)

for idx, row in important_featues_all_described.reset_index(drop=True).iterrows():
    pid = row[provider_id]
    cat = str(row['Category'])

    # ── Deterministic scheme labels from the dictionary (not from the narrative) ──
    scheme_results = derive_schemes(row['Impactful_Feature'], row['Impact_Score'], row['Scheme_Link'],
                                    row['Narrate_Role'], row['Concept_Group'],
                                    peer_pcts=row['peer_pct'], directions=row['Concern_Direction'])
    schemes_display, schemes_rationale = format_schemes_for_output(scheme_results)
    scheme_candidates = [r['scheme'] for r in scheme_results if r['scheme'] != 'other' and r.get('in_main')]
    _drv, _bg = shown_indices(row['Impact_Score'], row['Narrate_Role'])

    # ── Provider profile and peer comparison from df_scoring_master ──
    provider_profile = build_provider_profile(pid, cat, df_scoring_master)
    provider_feat_df = pd.DataFrame({'Feature': row['Feature']})
    baselines_text = format_baselines_for_prompt(cat, provider_feat_df, cohort_baselines, proc_cat_baselines, pid=pid)
    has_baselines = baselines_text != "Baselines not available."

    # ── Narrative (or evidence-only note) ──
    summary_text, info = generate_fraud_summary(
        provider_id=pid, category=cat,
        all_features=row['Impactful_Feature'], all_purposes=row['Impactful_Feature_Purpose'],
        all_scores=row['Impact_Score'], all_clinical_contexts=row['Clinical_Context'],
        decoded_codes=row['Code_Decoded'], provider_profile=provider_profile,
        baselines_text=baselines_text, narrate_roles=row['Narrate_Role'],
        brief_contexts=row['Brief_Context'], scheme_candidates=scheme_candidates)

    prompt_text = "\n".join(m['content'] for m in info['messages']) if info['messages'] else ""
    if info['status'] == 'narrative':
        # Numbers are grounded against the user message; ratios and value/median pairs only against the
        # Python-computed sentences (peer comparison, profile, value sentences of the features shown).
        value_text = "\n".join([baselines_text] + [row['Value_Sentence'][i] for i in _drv + _bg
                                                   if isinstance(row['Value_Sentence'][i], str)])
        qa_result = quality_check(summary_text, cat, has_baselines=has_baselines, provider_id=pid,
                                  prompt_text=info['messages'][1]['content'], allowed_schemes=scheme_candidates,
                                  value_text=value_text)
        status = qa_status(qa_result)
    else:
        qa_result, status = [info['status']], info['status']

    summary_rows.append({
        provider_id: pid,
        'Category': cat.upper(),
        'LLM_Summary': summary_text,
        'Detected_Schemes': schemes_display,
        'Scheme_Rationale': schemes_rationale,
        'QA_Flags': ', '.join(qa_result),
        'QA_Status': status,
        'Baselines_Available': has_baselines,
        'Scoring_Date': scoring_date,
        'Narrative_Features': ' | '.join(row['Impactful_Feature'][i] for i in _drv[:TIER1_DRIVERS]),
        'Notebook_Version': NOTEBOOK_VERSION,
        'Dictionary_Version': DICTIONARY_VERSION,
    })
    audit_rows.append({
        provider_id: pid, 'Category': cat.upper(), 'Scoring_Date': scoring_date,
        'Status': info['status'], 'Attempts': info['attempts'], 'New_Tokens': info['new_tokens'],
        'Hit_Token_Limit': info['hit_token_limit'], 'Parse': info['parse'],
        'Analysis_Chars': info['analysis_chars'],
        'Prompt_System': info['messages'][0]['content'] if info['messages'] else '',
        'Prompt_User': info['messages'][1]['content'] if info['messages'] else '',
        'Raw_Final': info['raw_final'],
        'Prompt_SHA1': hashlib.sha1(prompt_text.encode()).hexdigest() if prompt_text else '',
    })

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{_n_total} providers")

output_df = pd.DataFrame(summary_rows)
audit_df = pd.DataFrame(audit_rows)
print(f"\nDone. Total summaries: {len(output_df)}")
print(output_df['QA_Status'].value_counts().to_string())
_narr = output_df[output_df['QA_Status'].isin(['pass', 'needs_review'])]
if len(_narr):
    print(f"QA pass rate (narratives only): {(_narr['QA_Status'] == 'pass').mean():.1%}")
print("\nScheme distribution:")
print(output_df['Detected_Schemes'].value_counts().head(10))


##### Review QA Results

In [ ]:
_flag_counts = Counter(f.split(':')[0] + (':' + f.split(':')[1].split('(')[0] if f.startswith('banned:') else '')
                       for flags in output_df['QA_Flags'] for f in flags.split(', ') if f and f != 'pass')
print("QA flags across all notes:")
for k, v in _flag_counts.most_common():
    print(f"  {k}: {v}")

qa_failures = output_df[output_df['QA_Status'] == 'needs_review']
print(f"\nNotes needing review: {len(qa_failures)} / {len(output_df)}")
for _, row in qa_failures.head(5).iterrows():
    print(f"\n{'=' * 60}")
    print(f"Provider: {row[provider_id]} | Category: {row['Category']}")
    print(f"QA Flags: {row['QA_Flags']}")
    print(row['LLM_Summary'])


##### Save to CSV

In [ ]:
output_path = f'/Workspace/Users/{ROOT}/Projects/all_projects_predictions/data'

# Same file names as v3.4 so downstream reports keep working; the version is in the Notebook_Version column.
output_df.to_csv(
    f'{output_path}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony.csv',
    index=False
)
output_df.to_csv(
    f'{output_path}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony_{scoring_date}.csv',
    index=False
)
# Audit trail: exact prompt, raw model output and generation details for every provider
audit_df.to_json(f'{output_path}/llm_summary_audit_{scoring_date}.jsonl', orient='records', lines=True)
# Features behind each note (evidence card input)
features_narrated[[provider_id, 'Category', 'Feature', 'Impactful_Feature', 'Impact_Score', 'Narrate_Role',
                   'Validation_Status', 'Value_Sentence', 'Scheme_Link']].to_csv(
    f'{output_path}/llm_summary_features_{scoring_date}.csv', index=False)
print(f"Saved {len(output_df)} summaries, {len(audit_df)} audit records, {len(features_narrated)} feature rows.")


##### Save to Delta Table

In [ ]:
# Delta write. createDataFrame maps pandas columns by POSITION, so the frame is reordered to the schema first.
# The existing test table has an older schema; mergeSchema adds the new columns on append.
# Uncomment to run.

# spark = SparkSession.builder.appName("Referral_Processing").getOrCreate()
#
# schema = StructType([StructField(c, StringType(), True) for c in [
#     provider_id, 'Category', 'LLM_Summary', 'Detected_Schemes', 'Scheme_Rationale', 'QA_Flags',
#     'QA_Status', 'Baselines_Available', 'Scoring_Date', 'Narrative_Features', 'Notebook_Version',
#     'Dictionary_Version']])
#
# _out = output_df.copy()
# _out['Scoring_Date'] = pd.to_datetime(_out['Scoring_Date'], errors='coerce').dt.strftime('%Y-%m-%d')
# _out = _out[[f.name for f in schema.fields]].astype(str)
#
# if not _out.empty:
#     df_spark = spark.createDataFrame(_out, schema=schema)
#     if not spark.catalog.tableExists(test_table_llm_summarization):
#         df_spark.write.mode('overwrite').saveAsTable(test_table_llm_summarization)
#         print('Table created')
#     else:
#         df_spark.write.mode('append').option('mergeSchema', 'true').saveAsTable(test_table_llm_summarization)
#         print('Table appended')
# else:
#     print('No summaries to write')


##### Sample Output

In [ ]:
for _, row in output_df.head(3).iterrows():
    print(f"\n{'='*80}")
    print(f"Provider: {row[provider_id]} | {row['Category']}")
    print(f"Schemes: {row['Detected_Schemes']}")
    print(f"QA: {row['QA_Status']} ({row['QA_Flags']})")
    print(f"\n{row['LLM_Summary']}")


##### Trust Report

In [ ]:
# ── Trust report: read this before sharing the notes ──
print("=" * 70)
print(f"Run {NOTEBOOK_VERSION} | dictionary {DICTIONARY_VERSION} | scoring date {scoring_date}")
print(f"Features: {prep_stats['feature_rows']} rows for {prep_stats['providers']} provider-categories; "
      f"100% matched to a dictionary row (run stops otherwise)")
print(f"  removed: not narrated {prep_stats['dropped_not_narrated']}, on hold {prep_stats['dropped_on_hold']}, "
      f"not applicable {prep_stats['dropped_not_applicable']}, merged near-duplicates {prep_stats['merged_near_duplicates']}, "
      f"duplicate rows {prep_stats['dropped_duplicate_feature_rows']}, blank provider ID {prep_stats['dropped_blank_provider_id']}, "
      f"blank score {prep_stats['dropped_blank_score']}")
print(f"  driver levels moved to background (normal, below peers or flag absent): {prep_stats['drivers_moved_to_background']}")
_vs = prep_stats['with_value_sentence'] / max(prep_stats['feature_rows_narratable'], 1)
print(f"  narratable feature rows with a Python-computed value sentence: {_vs:.0%}")
print("Notes:", output_df['QA_Status'].value_counts().to_dict())
_narr = output_df[output_df['QA_Status'].isin(['pass', 'needs_review'])]
if len(_narr):
    print(f"  narratives passing QA: {(_narr['QA_Status'] == 'pass').mean():.0%} of {len(_narr)}")
_ung = output_df['QA_Flags'].str.contains('ungrounded_number|invented_ratio').sum()
print(f"  notes with a number or ratio not found in the prompt: {_ung}")
print(f"  generations that needed a retry: {(audit_df['Attempts'] > 1).sum()}; "
      f"fell back to evidence-only: {(audit_df['Status'] == 'generation_failed').sum()}")
_codes_seen = set(features_narrated['Feature'])
_no_desc = sorted({c for (kind, c) in CODES_WITHOUT_DESCRIPTION if any(c in f for f in _codes_seen)})
_group_only = sorted({c for (kind, c) in CODES_DESCRIBED_BY_GROUP if any(c in f for f in _codes_seen)})
print(f"Codes in narrated features described only by their ICD-10 group: {len(_group_only)}"
      + (f" (e.g. {', '.join(_group_only[:5])})" if _group_only else ""))
print(f"Codes in narrated features with no description at all (shown as the bare code): {len(_no_desc)}"
      + (f" (e.g. {', '.join(_no_desc[:10])})" if _no_desc else ""))
if UNKNOWN_LEVELS:
    print(f"Untranslated tier/band labels (shown to the LLM as 'an unusual level'): {sorted(UNKNOWN_LEVELS)}")
print("Share only notes with QA_Status = 'pass' or 'evidence_only' without review; read 'needs_review' notes first.")
print("=" * 70)
